# Stage 20: frozen-subspace causal planner steering

Stage 20 tests whether the action-consequence mechanism confirmed in Stages
18 and 19 forms a causal interface to downstream planning behavior.  It is a
fresh, non-visual evaluation: all decisions use numerical predictions,
rankings, selected actions, and simulator costs.  No example selection, video
scoring, or human visual judgment enters the protocol.

The exact Stage 18 block-4 bases and whitening transform remain frozen.  Stage
20 also requires the exact successful Stage 19 decision before proceeding.
There is no training, subspace refit, layer selection, coordinate reader,
Jacobian, JVP, VJP, or gradient computation.

For each fresh state, the untouched model produces a planner cost vector
(q(a)).  Its actions at baseline ranks 2, 3, and 4 are fixed as steering
targets, using no simulator outcome.  For target (t) and baseline-best donor
(b), a deterministic derangement \(\pi_t\) is frozen with
(\pi_t(t)=b).  The high-level counterfactual is therefore exact:

\[
q^{\mathrm{cf}}_t(a)=q(\pi_t(a)),
\qquad \arg\min_a q^{\mathrm{cf}}_t(a)=t.
\]

The primary rank-128 edit replaces the current projected action contrast with
the corresponding permuted donor contrast.  Rank 64 is a sensitivity.  Four
rank-matched random bases, the frozen shuffled-fit basis, wrong-state,
common-mode, complete-swap, and necessity-ablation conditions remain controls.

Two action families are evaluated separately: interleaved directions and the
pulsed equal-impulse profile.  Simulator truth is generated and physically
screened before model loading.  Baseline predictions may define targets, but
no intervention output or simulator cost may do so.  The primary behavioral
question is whether the learned edit moves target rank and chosen action
toward the exact counterfactual more than all matched controls.

Return `stage20_causal_planner_steering_result_bundle_<signature>.zip`.

In [ ]:
# SINGLE CONFIGURATION BLOCK
# Create these Colab secrets for a source-bound pilot:
# STAGE20_RUN_MODE=pilot
# STAGE20_SOURCE_COMMIT=<full 40-hex commit shown in the handoff>
# STAGE20_RUN_NONCE=<a new unique label, for example steering_20260804_a>
# The Stage 18 subspace and Stage 19 decision paths have successful-run defaults.
RUN_MODE = "smoke"
EXPERIMENT_SOURCE_REF = ""
RUN_NONCE = "smoke"
STAGE18_SUBSPACE_PATH = (
    "/content/drive/MyDrive/counterfactual_faithfulness_stage18_rank64/"
    "pilot_f1b34beffcac/subspaces/frozen_rank64_confirmation_subspaces.npz"
)
STAGE19_DECISION_PATH = (
    "/content/drive/MyDrive/counterfactual_faithfulness_stage19_transfer/"
    "pilot_b7f2b6cef37f/stage19_decision.json"
)
try:
    from google.colab import userdata as _colab_userdata

    RUN_MODE = str(_colab_userdata.get("STAGE20_RUN_MODE") or RUN_MODE).strip().lower()
    EXPERIMENT_SOURCE_REF = str(
        _colab_userdata.get("STAGE20_SOURCE_COMMIT") or EXPERIMENT_SOURCE_REF
    ).strip()
    RUN_NONCE = str(_colab_userdata.get("STAGE20_RUN_NONCE") or RUN_NONCE).strip()
    STAGE18_SUBSPACE_PATH = str(
        _colab_userdata.get("STAGE20_STAGE18_SUBSPACE_PATH") or STAGE18_SUBSPACE_PATH
    ).strip()
    STAGE19_DECISION_PATH = str(
        _colab_userdata.get("STAGE20_STAGE19_DECISION_PATH") or STAGE19_DECISION_PATH
    ).strip()
except Exception:
    pass

if RUN_MODE == "pilot":
    if RUN_NONCE in {"", "smoke"}:
        raise ValueError("pilot mode requires a unique STAGE20_RUN_NONCE")
    if not all(value.isalnum() or value in "-_" for value in RUN_NONCE):
        raise ValueError("STAGE20_RUN_NONCE may contain only letters, numbers, '-' and '_'")

MOUNT_DRIVE = True
DOWNLOAD_RESULTS = True
CONTINUE_AFTER_BENCHMARK = True
MAX_ESTIMATED_TOTAL_MINUTES = 90.0
FRESH_RUN_REQUIRED = True

OUTPUT_DIR = "/content/counterfactual_faithfulness_stage20_steering"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/counterfactual_faithfulness_stage20_steering"

PROTOCOL_ID = "stage20-frozen-subspace-causal-planner-steering-v1"
NOTEBOOK_PROTOCOL_SHA256 = "8d46d044cd0c181fe0443447a2242120e768dae674ce5bd05ed304aa14166f8b"
EVIDENCE_STATUS = "CONFIRMATORY_ONLY_IF_SOURCE_BOUND_FRESH_AND_PRIOR_ARTIFACTS_BOUND"
EXPERIMENT_REPOSITORY = "grewalsk/counterfactual-faithfulness-research"
EXPERIMENT_NOTEBOOK_PATH = "notebooks/20_causal_planner_steering.ipynb"
EXPERIMENT_BUILDER_PATH = "notebooks/build_stage20_causal_planner_steering_notebook.py"
EXPERIMENT_NUMERICAL_PATH = "src/cf_faithfulness/stage20_planner_steering.py"

SEED = 20101
DESIGN_SEED = 20137
MODEL_NAME = "jepa_wm_pusht"
ENVIRONMENT = "PushT"
FRAMESKIP = 5
PRIMARY_HORIZON = 3
TARGET_STEPS = [PRIMARY_HORIZON]
FIXED_BLOCK = 4
ACTIVE_BLOCKS = [FIXED_BLOCK]
EXPECTED_CARRIER_CHANNELS = 400

EXPECTED_STAGE18_SUBSPACE_SHA256 = "2f9c496d54623a9062e465a18c70039acc18cb8a1cc2833a5f4ade162ca3f90b"
EXPECTED_STAGE18_SOURCE_COMMIT = "16edd247cddcb1aa121340eb5fa42bd9e07004c3"
EXPECTED_STAGE18_STATUS = "CONFIRMED_BIDIRECTIONAL_RANK64_MEDIATOR"
EXPECTED_STAGE18_AMBIENT_DIMENSION = 102400
EXPECTED_STAGE18_MAX_RANK = 128
EXPECTED_STAGE19_DECISION_SHA256 = "493fdf5c707189caea11043db7d208dbc38677dcf5881008e13bede87f40be9c"
EXPECTED_STAGE19_SOURCE_IDENTITY_SHA256 = "6fad7d1ee14efa0898125faaa4500a2ab7b62d81591412364b4379c43ec9ffcf"
EXPECTED_STAGE19_SOURCE_COMMIT = "bf8c3950fc1112b38baa2453e39793592537ec47"
EXPECTED_STAGE19_STATUS = "CONFIRMED_TRANSFER_ALL_UNSEEN_ACTION_FAMILIES"

TRANSFER_FAMILIES = ["rotated_direction", "pulsed_equal_impulse"]
EVALUATION_POOL_TRAJECTORIES = list(range(600, 680))
EVALUATION_TRAJECTORY_TARGET_PER_FAMILY = 32
TARGET_BASELINE_RANKS = [1, 2, 3]
STATES_PER_TRAJECTORY = 1
TASK_ID_OFFSET = 2000
ACTIONS_PER_STATE = 13
ACTION_STEPS = PRIMARY_HORIZON * FRAMESKIP
APPROACH_DISTANCE = 80.0
MIN_ELIGIBLE_COST_SPREAD = 0.02
MIN_ELIGIBLE_NON_TIED_PAIR_FRACTION = 0.20
MIN_ELIGIBLE_CONTACT_BRANCHES = 2
PHYSICAL_COST_TIE = 1e-4

OUTPUT_SKETCH_DIM = 256
TRAIN_OUTPUT_SKETCH_SEED = 18161
EVAL_OUTPUT_SKETCH_SEED = 18183
PRIMARY_STEERING_RANK = 128
SENSITIVITY_RANK = 64
PERMUTATION_SEED = 20251
BOOTSTRAP_SEED = 20269
CAUSAL_RANDOM_DRAWS = 4
STEERING_DOSES = [0.5, 1.0]
BOOTSTRAP_DRAWS = 10000
INTERVENTION_FORWARDS_PER_RECORD = 39
RESULT_ROWS_PER_RECORD = 54

MIN_FULL_SWAP_COEFFICIENT = 0.80
MIN_FULL_TARGET_CHOICE_RATE = 0.90
MIN_PRIMARY_OUTPUT_COEFFICIENT = 0.25
MIN_OUTPUT_GAIN_OVER_RANDOM = 0.10
MIN_TARGET_RANK_GAIN_OVER_RANDOM = 0.25
MIN_CHOICE_MATCH_GAIN_OVER_RANDOM = 0.05
MIN_NECESSITY_REDUCTION = 0.03
MIN_NECESSITY_GAIN_OVER_RANDOM = 0.02
MIN_NECESSITY_GAIN_OVER_SHUFFLED = 0.02
MAX_ZERO_EDIT_ERROR = 1e-6

if RUN_MODE == "smoke":
    ACTIVE_EVALUATION_POOL_TRAJECTORIES = EVALUATION_POOL_TRAJECTORIES[:8]
    ACTIVE_EVALUATION_TARGET_PER_FAMILY = 2
    ACTIVE_TARGET_BASELINE_RANKS = [1]
    ACTIVE_CAUSAL_RANDOM_DRAWS = 1
    ACTIVE_STEERING_DOSES = [1.0]
    ACTIVE_BOOTSTRAP_DRAWS = 64
    ACTIVE_INTERVENTION_FORWARDS_PER_RECORD = 10
    ACTIVE_RESULT_ROWS_PER_RECORD = 11
elif RUN_MODE == "pilot":
    ACTIVE_EVALUATION_POOL_TRAJECTORIES = EVALUATION_POOL_TRAJECTORIES
    ACTIVE_EVALUATION_TARGET_PER_FAMILY = EVALUATION_TRAJECTORY_TARGET_PER_FAMILY
    ACTIVE_TARGET_BASELINE_RANKS = TARGET_BASELINE_RANKS
    ACTIVE_CAUSAL_RANDOM_DRAWS = CAUSAL_RANDOM_DRAWS
    ACTIVE_STEERING_DOSES = STEERING_DOSES
    ACTIVE_BOOTSTRAP_DRAWS = BOOTSTRAP_DRAWS
    ACTIVE_INTERVENTION_FORWARDS_PER_RECORD = INTERVENTION_FORWARDS_PER_RECORD
    ACTIVE_RESULT_ROWS_PER_RECORD = RESULT_ROWS_PER_RECORD
else:
    raise ValueError(
        "STAGE20_RUN_MODE must contain only smoke or pilot; "
        f"received {RUN_MODE!r}"
    )

REPO_URL = "https://github.com/facebookresearch/jepa-wms.git"
REPO_COMMIT = "13cf1d9c7e476f53c17714d2e0f1dc239a883ce0"
EXPECTED_HF_REVISION = "9b9c41ef249466630dbf1a20e78391865d07b3b9"
EXPECTED_PRETRAINED_ASSET_SHA256 = {
    "jepa_wm_pusht.pth.tar": "9beca3eafe0739c3b3adb5d734fa435ccbda0fea8a65d53d4cccec176aaaa0eb",
    "dinov2_vits14_pretrain.pth": "b938bf1bc15cd2ec0feacfe3a1bb553fe8ea9ca46a7e1d8d00217f29aef60cd9",
}
ASSET_REPOSITORY = "grewalsk/counterfactual-faithfulness-research"
ASSET_COMMIT = "2326e74556f6f81db2560e4396f4cc52c16a28f4"
ASSET_SPECS = {
    "physical_decoders.pt": {
        "path": "results/bundles/stage12_result_bundle/frozen_training_decoders/jepa_wm_pusht_f975a0a746e7_training_decoders.pt",
        "sha256": "51b2dbb0a81df432a2db5b941de83717e9979e761d57365f47d93d2dd0c0c694",
    },
}

assert ACTIONS_PER_STATE == 13
assert ACTION_STEPS == 15
assert FIXED_BLOCK == 4
assert PRIMARY_STEERING_RANK == 128
assert SENSITIVITY_RANK == 64
assert len(TRANSFER_FAMILIES) == 2
assert TARGET_BASELINE_RANKS == [1, 2, 3]


PROTOCOL_CONFIG_KEYS = ['RUN_MODE', 'EXPERIMENT_SOURCE_REF', 'RUN_NONCE', 'STAGE18_SUBSPACE_PATH', 'STAGE19_DECISION_PATH', 'MOUNT_DRIVE', 'DOWNLOAD_RESULTS', 'CONTINUE_AFTER_BENCHMARK', 'MAX_ESTIMATED_TOTAL_MINUTES', 'FRESH_RUN_REQUIRED', 'OUTPUT_DIR', 'DRIVE_OUTPUT_DIR', 'PROTOCOL_ID', 'NOTEBOOK_PROTOCOL_SHA256', 'EVIDENCE_STATUS', 'EXPERIMENT_REPOSITORY', 'EXPERIMENT_NOTEBOOK_PATH', 'EXPERIMENT_BUILDER_PATH', 'EXPERIMENT_NUMERICAL_PATH', 'SEED', 'DESIGN_SEED', 'MODEL_NAME', 'ENVIRONMENT', 'FRAMESKIP', 'PRIMARY_HORIZON', 'TARGET_STEPS', 'FIXED_BLOCK', 'ACTIVE_BLOCKS', 'EXPECTED_CARRIER_CHANNELS', 'EXPECTED_STAGE18_SUBSPACE_SHA256', 'EXPECTED_STAGE18_SOURCE_COMMIT', 'EXPECTED_STAGE18_STATUS', 'EXPECTED_STAGE18_AMBIENT_DIMENSION', 'EXPECTED_STAGE18_MAX_RANK', 'EXPECTED_STAGE19_DECISION_SHA256', 'EXPECTED_STAGE19_SOURCE_IDENTITY_SHA256', 'EXPECTED_STAGE19_SOURCE_COMMIT', 'EXPECTED_STAGE19_STATUS', 'TRANSFER_FAMILIES', 'EVALUATION_POOL_TRAJECTORIES', 'EVALUATION_TRAJECTORY_TARGET_PER_FAMILY', 'TARGET_BASELINE_RANKS', 'STATES_PER_TRAJECTORY', 'TASK_ID_OFFSET', 'ACTIONS_PER_STATE', 'ACTION_STEPS', 'APPROACH_DISTANCE', 'MIN_ELIGIBLE_COST_SPREAD', 'MIN_ELIGIBLE_NON_TIED_PAIR_FRACTION', 'MIN_ELIGIBLE_CONTACT_BRANCHES', 'PHYSICAL_COST_TIE', 'OUTPUT_SKETCH_DIM', 'TRAIN_OUTPUT_SKETCH_SEED', 'EVAL_OUTPUT_SKETCH_SEED', 'PRIMARY_STEERING_RANK', 'SENSITIVITY_RANK', 'PERMUTATION_SEED', 'BOOTSTRAP_SEED', 'CAUSAL_RANDOM_DRAWS', 'STEERING_DOSES', 'BOOTSTRAP_DRAWS', 'INTERVENTION_FORWARDS_PER_RECORD', 'RESULT_ROWS_PER_RECORD', 'MIN_FULL_SWAP_COEFFICIENT', 'MIN_FULL_TARGET_CHOICE_RATE', 'MIN_PRIMARY_OUTPUT_COEFFICIENT', 'MIN_OUTPUT_GAIN_OVER_RANDOM', 'MIN_TARGET_RANK_GAIN_OVER_RANDOM', 'MIN_CHOICE_MATCH_GAIN_OVER_RANDOM', 'MIN_NECESSITY_REDUCTION', 'MIN_NECESSITY_GAIN_OVER_RANDOM', 'MIN_NECESSITY_GAIN_OVER_SHUFFLED', 'MAX_ZERO_EDIT_ERROR', 'REPO_URL', 'REPO_COMMIT', 'EXPECTED_HF_REVISION', 'EXPECTED_PRETRAINED_ASSET_SHA256', 'ASSET_REPOSITORY', 'ASSET_COMMIT', 'ASSET_SPECS']

In [ ]:
import subprocess
import sys

# Preserve Colab's CUDA-matched torch and torchvision.
PINNED = [
    "einops==0.8.1",
    "tensordict==0.9.1",
    "timm==1.0.19",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "PyYAML==6.0.2",
    "huggingface_hub==0.36.2",
    "hf-xet==1.5.1",
    "gym==0.23.1",
    "pygame==2.6.1",
    "pymunk==6.8.0",
    "opencv-python-headless==4.11.0.86",
    "shapely==2.1.2",
    "lpips==0.1.4",
    "ruamel.yaml==0.18.10",
    "scikit-learn==1.6.1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PINNED],
    check=True,
)
print("Installed pinned non-PyTorch dependencies.")

In [ ]:
import csv
import gc
import hashlib
import json
import logging
import math
import os
import platform
import random
import resource
import shutil
import subprocess
import sys
import time
import traceback
import urllib.request
from collections import defaultdict
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as torch_functional
import torchvision
import yaml


def ensure_colab_drive():
    from google.colab import drive

    mountpoint = "/content/drive"
    if Path(mountpoint, "MyDrive").is_dir():
        return
    drive.mount(mountpoint, timeout_ms=600_000)
    if not Path(mountpoint, "MyDrive").is_dir():
        raise RuntimeError("Google Drive mount did not produce MyDrive")


if MOUNT_DRIVE:
    ensure_colab_drive()
    OUTPUT_DIR = DRIVE_OUTPUT_DIR

def build_protocol_config(namespace, pinned):
    missing = [key for key in PROTOCOL_CONFIG_KEYS if key not in namespace]
    if missing:
        raise RuntimeError(f"missing frozen protocol configuration: {missing}")
    config = {key: namespace[key] for key in PROTOCOL_CONFIG_KEYS}
    config["PROTOCOL_CONFIG_KEYS"] = list(PROTOCOL_CONFIG_KEYS)
    config["PINNED"] = list(pinned)
    # Fail here with a protocol-specific error if a declared value ever ceases
    # to be JSON-safe.  Ambient notebook globals are deliberately unreachable.
    try:
        json.dumps(config, sort_keys=True, allow_nan=False)
    except (TypeError, ValueError) as error:
        raise RuntimeError(
            "a declared Stage 20 protocol value is not JSON serializable"
        ) from error
    return config


CONFIG = build_protocol_config(globals(), PINNED)
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG, sort_keys=True, allow_nan=False).encode()
).hexdigest()
OUT = Path(OUTPUT_DIR) / f"{RUN_MODE}_{RUN_SIGNATURE[:12]}"
OUT_PREEXISTED = OUT.exists()
if RUN_MODE == "pilot" and FRESH_RUN_REQUIRED and OUT_PREEXISTED:
    raise RuntimeError("fresh pilot output already exists; choose a new STAGE20_RUN_NONCE")
ASSET_DIR = OUT / "assets"
DESIGN_DIR = OUT / "design"
TRUTH_DIR = OUT / "truth"
BASELINE_DIR = OUT / "baseline_shards"
SUBSPACE_DIR = OUT / "subspaces"
ANALYSIS_DIR = OUT / "analysis"
INTERVENTION_DIR = OUT / "intervention_shards"
EVIDENCE_DIR = OUT / "evaluation_evidence"
PLOT_DIR = OUT / "plots"
LOG_DIR = OUT / "logs"
for directory in [
    OUT, ASSET_DIR, DESIGN_DIR, TRUTH_DIR, BASELINE_DIR, SUBSPACE_DIR,
    ANALYSIS_DIR, INTERVENTION_DIR, EVIDENCE_DIR, PLOT_DIR, LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = (
    Path("/content/drive/MyDrive/cf_faithfulness_cache")
    if MOUNT_DRIVE
    else Path("/content/cf_faithfulness_cache")
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["MPLCONFIGDIR"] = str(CACHE_ROOT / "matplotlib")
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["JEPAWM_LOGS"] = str(CACHE_ROOT / "unused_jepawm_logs")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU")
if tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) < (2, 7):
    raise RuntimeError(f"JEPA-WMs requires torch>=2.7; found {torch.__version__}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "run.log"),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,
)
log = logging.getLogger("stage20_steering")


def write_json(path, payload):
    temporary = Path(path).with_suffix(".tmp.json")
    temporary.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def write_csv(path, rows):
    if not rows:
        return
    temporary = Path(path).with_suffix(".tmp.csv")
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def atomic_npz(path, **arrays):
    temporary = Path(str(path) + ".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_bytes):
            digest.update(chunk)
    return digest.hexdigest()


def download_asset(name):
    specification = ASSET_SPECS[name]
    destination = ASSET_DIR / name
    if destination.exists() and sha256_file(destination) == specification["sha256"]:
        return destination
    destination.unlink(missing_ok=True)
    url = (
        "https://raw.githubusercontent.com/"
        f"{ASSET_REPOSITORY}/{ASSET_COMMIT}/{specification['path']}"
    )
    temporary = destination.with_suffix(destination.suffix + ".part")
    urllib.request.urlretrieve(url, temporary)
    observed = sha256_file(temporary)
    if observed != specification["sha256"]:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"{name} hash mismatch: {observed}")
    temporary.replace(destination)
    return destination


REMOTE_NOTEBOOK_CODE_CELLS = []


def canonical_cell_source(value):
    return str(value).replace("\r\n", "\n").strip()


def source_identity():
    global REMOTE_NOTEBOOK_CODE_CELLS
    payload = {
        "protocol_id": PROTOCOL_ID,
        "notebook_protocol_sha256": NOTEBOOK_PROTOCOL_SHA256,
        "repository": EXPERIMENT_REPOSITORY,
        "source_ref": EXPERIMENT_SOURCE_REF,
        "execution_verified": False,
    }
    if not EXPERIMENT_SOURCE_REF:
        payload["status"] = "UNBOUND_EXPLORATORY_NOTEBOOK"
        payload["confirmation_eligible"] = False
        return payload
    source_ref = EXPERIMENT_SOURCE_REF.lower()
    if len(source_ref) != 40 or any(value not in "0123456789abcdef" for value in source_ref):
        raise RuntimeError("STAGE20_SOURCE_COMMIT must be a full 40-hex commit")
    payload["resolved_commit"] = source_ref
    base = (
        "https://raw.githubusercontent.com/"
        f"{EXPERIMENT_REPOSITORY}/{source_ref}/"
    )
    payload["files"] = {}
    for label, relative in [
        ("notebook", EXPERIMENT_NOTEBOOK_PATH),
        ("builder", EXPERIMENT_BUILDER_PATH),
        ("numerical", EXPERIMENT_NUMERICAL_PATH),
    ]:
        with urllib.request.urlopen(base + relative) as response:
            content = response.read()
        payload["files"][label] = {
            "path": relative,
            "sha256": hashlib.sha256(content).hexdigest(),
            "size_bytes": len(content),
        }
        if label == "notebook":
            remote_notebook = json.loads(content.decode())
            REMOTE_NOTEBOOK_CODE_CELLS = [
                canonical_cell_source("".join(cell.get("source", [])))
                for cell in remote_notebook["cells"]
                if cell.get("cell_type") == "code"
            ]
            payload["remote_code_cells"] = len(REMOTE_NOTEBOOK_CODE_CELLS)
    payload["status"] = "SOURCE_BOUND_EXECUTION_UNVERIFIED"
    payload["confirmation_eligible"] = False
    return payload


def verify_executed_notebook_through(cell_header):
    if SOURCE_IDENTITY["status"] == "UNBOUND_EXPLORATORY_NOTEBOOK":
        return False
    expected_index = next(
        index
        for index, source in enumerate(REMOTE_NOTEBOOK_CODE_CELLS)
        if source.startswith(cell_header)
    )
    expected = REMOTE_NOTEBOOK_CODE_CELLS[: expected_index + 1]
    shell = get_ipython()
    history = [
        canonical_cell_source(value)
        for value in shell.user_ns.get("_ih", [])[1:]
        if canonical_cell_source(value)
    ]
    matched = len(history) >= len(expected) and history[-len(expected) :] == expected
    prefix_payload = json.dumps(expected, ensure_ascii=False, separators=(",", ":"))
    SOURCE_IDENTITY["execution_verified"] = bool(matched)
    SOURCE_IDENTITY["executed_through_code_cell"] = expected_index
    SOURCE_IDENTITY["executed_through_header"] = cell_header
    SOURCE_IDENTITY["executed_code_cells_verified"] = len(expected)
    SOURCE_IDENTITY["executed_cells_sha256"] = hashlib.sha256(
        prefix_payload.encode()
    ).hexdigest()
    SOURCE_IDENTITY["status"] = (
        "SOURCE_BOUND_EXECUTION_VERIFIED"
        if matched else "SOURCE_BOUND_EXECUTION_MISMATCH"
    )
    SOURCE_IDENTITY["confirmation_eligible"] = bool(matched)
    write_json(OUT / "source_identity.json", SOURCE_IDENTITY)
    if not matched:
        raise RuntimeError(
            "executed code is not the exact committed notebook prefix; "
            "restart and Run all from the source-bound artifact"
        )
    return True


VERSIONS = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_total_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
}
SOURCE_IDENTITY = source_identity()
write_json(OUT / "config.json", {**CONFIG, "run_signature": RUN_SIGNATURE})
write_json(OUT / "versions.json", VERSIONS)
write_json(OUT / "source_identity.json", SOURCE_IDENTITY)
(OUT / "FAILURE_TRACE.txt").write_text("PENDING\n")

TIMINGS = {}
MEMORY = []
PROVENANCE_COUNTS = {"truth_generated": 0, "baseline_generated": 0, "intervention_generated": 0, "patched_forwards_generated": 0, "cache_hits": 0}
PIPELINE_FAILED = False
FAILURE_MESSAGE = ""


def record_failure(stage):
    global PIPELINE_FAILED, FAILURE_MESSAGE
    PIPELINE_FAILED = True
    FAILURE_MESSAGE = f"STAGE: {stage}\n{traceback.format_exc()}"
    (OUT / "FAILURE_TRACE.txt").write_text(FAILURE_MESSAGE)
    log.exception("Captured failure in %s", stage)


def memory_report(stage):
    maximum_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    row = {
        "stage": stage,
        "cpu_peak_gib": float(maximum_rss * 1024 / 2**30),
        "gpu_allocated_gib": float(torch.cuda.memory_allocated() / 2**30),
        "gpu_reserved_gib": float(torch.cuda.memory_reserved() / 2**30),
        "gpu_peak_allocated_gib": float(torch.cuda.max_memory_allocated() / 2**30),
    }
    MEMORY.append(row)
    write_json(OUT / "memory.json", MEMORY)
    return row


for asset_name in ASSET_SPECS:
    download_asset(asset_name)
print(json.dumps(VERSIONS, indent=2))
print(json.dumps(SOURCE_IDENTITY, indent=2))
print(f"Durable run directory: {OUT}")
memory_report("startup")

In [ ]:
def array_sha256(value):
    """Hash an array together with its dtype and shape."""
    array = np.ascontiguousarray(value)
    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode())
    digest.update(str(array.shape).encode())
    digest.update(array.tobytes())
    return digest.hexdigest()




def transform_primal_channels(values, inverse_square_root):
    """Whiten hidden-space vectors along their final channel dimension."""
    array = np.asarray(values, dtype=np.float64)
    inverse = np.asarray(inverse_square_root, dtype=np.float64)
    if array.shape[-1] != inverse.shape[0] or inverse.shape[0] != inverse.shape[1]:
        raise ValueError("channel metric does not match primal values")
    return np.einsum("...c,dc->...d", array, inverse, optimize=True)


def inverse_transform_primal_channels(values, square_root):
    """Map a whitened hidden-space vector back to native coordinates."""
    array = np.asarray(values, dtype=np.float64)
    root = np.asarray(square_root, dtype=np.float64)
    if array.shape[-1] != root.shape[0] or root.shape[0] != root.shape[1]:
        raise ValueError("channel metric does not match primal values")
    return np.einsum("...c,dc->...d", array, root, optimize=True)


class CountSketchProjector:
    """Deterministic norm-stabilized projection for context/decoder features."""

    def __init__(self, input_dim, output_dim, seed, device="cuda"):
        rng = np.random.default_rng(int(seed))
        bucket = rng.integers(0, output_dim, size=input_dim, dtype=np.int64)
        sign = rng.choice(np.asarray([-1.0, 1.0], dtype=np.float32), input_dim)
        counts = np.bincount(bucket, minlength=output_dim).astype(np.float32)
        counts[counts == 0] = 1.0
        self.bucket = torch.as_tensor(bucket, device=device, dtype=torch.long)
        self.sign = torch.as_tensor(sign, device=device, dtype=torch.float32)
        self.scale = torch.as_tensor(np.sqrt(counts), device=device)
        self.output_dim = int(output_dim)

    def __call__(self, values):
        values = values.float().flatten(1)
        output = torch.zeros(
            values.shape[0], self.output_dim, device=values.device
        )
        output.scatter_add_(
            1,
            self.bucket[None].expand(values.shape[0], -1),
            values * self.sign[None],
        )
        return output / self.scale[None]










def candidate_center(values):
    """Remove the within-state mean across candidate actions."""
    array = np.asarray(values, dtype=np.float64)
    if array.ndim < 2 or array.shape[0] < 2:
        raise ValueError("values must have candidate rows and at least two actions")
    return array - np.mean(array, axis=0, keepdims=True)


def stable_seed(*items):
    """Derive a deterministic unsigned 32-bit seed from structured labels."""
    payload = "|".join(str(item) for item in items).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little") % (2**32)


def fixed_derangement(length, seed):
    """Return a deterministic permutation with no fixed points."""
    length = int(length)
    if length < 2:
        raise ValueError("a derangement requires at least two items")
    rng = np.random.default_rng(int(seed))
    identity = np.arange(length, dtype=np.int64)
    for _ in range(10_000):
        candidate = rng.permutation(length)
        if np.all(candidate != identity):
            return candidate
    # This path is effectively unreachable, but a fixed cyclic permutation is
    # a valid fail-closed fallback for every length greater than one.
    return np.roll(identity, 1)




def action_swap_delta(values, permutation, basis=None, dose=1.0):
    """Construct a finite donor-action residual edit.

    ``values`` has one activation row per candidate action.  ``basis`` contains
    orthonormal column directions.  With ``basis=None`` and ``dose=1``, adding
    the returned edit exactly permutes the complete candidate activations.
    """
    array = np.asarray(values, dtype=np.float64)
    original_shape = array.shape
    flat = array.reshape(array.shape[0], -1)
    permutation = np.asarray(permutation, dtype=np.int64)
    if permutation.shape != (len(flat),) or sorted(permutation.tolist()) != list(
        range(len(flat))
    ):
        raise ValueError("permutation is invalid")
    residual = candidate_center(flat)
    difference = residual[permutation] - residual
    if basis is not None:
        directions = np.asarray(basis, dtype=np.float64)
        if directions.ndim != 2 or directions.shape[0] != flat.shape[1]:
            raise ValueError("basis does not match flattened activation width")
        difference = (difference @ directions) @ directions.T
    return (float(dose) * difference).reshape(original_shape)


def matched_common_mode(template_delta, direction):
    """Repeat one direction across actions and exactly match Frobenius energy."""
    template = np.asarray(template_delta, dtype=np.float64)
    vector = np.asarray(direction, dtype=np.float64).reshape(-1)
    if template.ndim < 2 or vector.size != int(np.prod(template.shape[1:])):
        raise ValueError("common direction does not match the activation shape")
    norm = np.linalg.norm(vector)
    if norm <= 1e-12:
        raise ValueError("common direction must be nonzero")
    repeated = np.broadcast_to(vector / norm, (template.shape[0], vector.size)).copy()
    repeated *= np.linalg.norm(template) / max(np.linalg.norm(repeated), 1e-12)
    return repeated.reshape(template.shape)


def donor_transfer_metrics(baseline, patched, permutation):
    """Measure directional and reconstructive transfer toward donor outcomes.

    All calculations occur after candidate centering, so shared output drift
    cannot masquerade as donor-specific transfer.  The no-edit baseline has
    coefficient/reconstruction zero; an exact donor permutation has one.
    """
    base = np.asarray(baseline, dtype=np.float64).reshape(len(baseline), -1)
    edit = np.asarray(patched, dtype=np.float64).reshape(len(patched), -1)
    permutation = np.asarray(permutation, dtype=np.int64)
    if base.shape != edit.shape or permutation.shape != (len(base),):
        raise ValueError("donor-transfer inputs have inconsistent shapes")
    centered_base = candidate_center(base)
    centered_edit = candidate_center(edit)
    target = centered_base[permutation] - centered_base
    observed = centered_edit - centered_base
    denominator = float(np.sum(target**2))
    observed_energy = float(np.sum(observed**2))
    if denominator <= 1e-12:
        return {
            "target_energy": denominator,
            "coefficient": math.nan,
            "cosine": math.nan,
            "reconstruction": math.nan,
            "mean_shift_ratio": math.nan,
        }
    coefficient = float(np.sum(observed * target) / denominator)
    cosine_denominator = math.sqrt(denominator * observed_energy)
    cosine = (
        float(np.sum(observed * target) / cosine_denominator)
        if cosine_denominator > 1e-12
        else 0.0
    )
    reconstruction = 1.0 - float(np.sum((observed - target) ** 2)) / denominator
    mean_shift = np.mean(edit, axis=0) - np.mean(base, axis=0)
    return {
        "target_energy": denominator,
        "coefficient": coefficient,
        "cosine": cosine,
        "reconstruction": reconstruction,
        "mean_shift_ratio": float(np.linalg.norm(mean_shift) / math.sqrt(denominator)),
    }








def pair_indices(length):
    return np.triu_indices(int(length), k=1)


def ranking_metrics(true_cost, predicted_cost, tie=1e-9):
    """Return planning regret and pairwise-ranking outcomes for one state."""
    truth = np.asarray(true_cost, dtype=np.float64)
    prediction = np.asarray(predicted_cost, dtype=np.float64)
    if truth.shape != prediction.shape or truth.ndim != 1:
        raise ValueError("cost vectors must be aligned and one-dimensional")
    selected = int(np.argmin(prediction))
    oracle = int(np.argmin(truth))
    best = float(np.min(truth))
    chosen = float(truth[selected])
    spread = float(np.max(truth) - best)
    normalized_regret = (chosen - best) / spread if spread > tie else 0.0
    left, right = pair_indices(len(truth))
    true_margin = truth[left] - truth[right]
    predicted_margin = prediction[left] - prediction[right]
    valid = np.abs(true_margin) > tie
    credit = np.full(len(left), np.nan)
    same = np.sign(true_margin) == np.sign(predicted_margin)
    credit[valid & same] = 1.0
    credit[valid & (np.abs(predicted_margin) <= tie)] = 0.5
    credit[valid & np.isnan(credit)] = 0.0
    weights = np.abs(true_margin)
    weighted = (
        float(np.nansum(weights * credit) / np.sum(weights[valid]))
        if np.any(valid)
        else math.nan
    )
    return {
        "selected_action": selected,
        "oracle_action": oracle,
        "top1_correct": float(chosen <= best + tie),
        "normalized_regret": float(normalized_regret),
        "weighted_pairwise_accuracy": weighted,
    }


def pose_target(states):
    """Map PushT simulator states to normalized block pose coordinates."""
    states = np.asarray(states, dtype=np.float64)
    angle = states[..., 4]
    return np.stack(
        [states[..., 2] / 512.0, states[..., 3] / 512.0, np.sin(angle), np.cos(angle)],
        axis=-1,
    )


def decoded_task_cost(prediction, goal):
    """Stage 3/4 normalized PushT goal cost for decoded block poses."""
    prediction = np.asarray(prediction, dtype=np.float64)
    goal = np.asarray(goal, dtype=np.float64)
    angle = np.arctan2(prediction[..., 2], prediction[..., 3])
    angular = np.arctan2(np.sin(angle - goal[2]), np.cos(angle - goal[2]))
    return np.linalg.norm(
        np.concatenate(
            [prediction[..., :2] - goal[:2] / 512.0, (angular / np.pi)[..., None]],
            axis=-1,
        ),
        axis=-1,
    )


def exact_positive_sign_test(values):
    """One-sided exact sign-test p-value after dropping exact zeros."""
    array = np.asarray(values, dtype=np.float64)
    array = array[np.isfinite(array) & (array != 0)]
    positives = int(np.sum(array > 0))
    total = int(len(array))
    if total == 0:
        return {"positive": 0, "nonzero": 0, "p_value": math.nan}
    probability = sum(math.comb(total, k) for k in range(positives, total + 1)) / 2**total
    return {"positive": positives, "nonzero": total, "p_value": float(probability)}


def norm_match(candidate, reference):
    value = np.asarray(candidate, dtype=np.float64)
    target = np.asarray(reference, dtype=np.float64)
    norm = np.linalg.norm(value)
    if norm <= 1e-12:
        raise RuntimeError("cannot norm-match a zero control")
    return value * (np.linalg.norm(target) / norm)


def clustered_bootstrap_mean(values, groups, draws, seed):
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups)
    unique = np.unique(groups)
    by_group = {group: values[groups == group] for group in unique}
    rng = np.random.default_rng(int(seed))
    results = np.empty(int(draws), dtype=np.float64)
    for index in range(int(draws)):
        sampled = rng.choice(unique, size=len(unique), replace=True)
        results[index] = np.mean([np.mean(by_group[group]) for group in sampled])
    return results


# Algebraic identities execute on CPU before any simulator or model access.
_rng = np.random.default_rng(1701)
_toy = _rng.normal(size=(7, 3, 5))
_permutation = fixed_derangement(len(_toy), 1703)
_full = _toy + action_swap_delta(_toy, _permutation)
if not np.allclose(_full, _toy[_permutation], atol=1e-12):
    raise AssertionError("full finite action swap identity failed")
_metrics = donor_transfer_metrics(_toy, _toy[_permutation], _permutation)
if not np.isclose(_metrics["coefficient"], 1.0, atol=1e-12):
    raise AssertionError("donor transfer coefficient identity failed")




def projection_ablation_delta(values, basis, dose=1.0):
    """Remove an action-centered component lying in ``span(basis)``.

    At dose one, adding the returned delta to ``values`` leaves the shared
    candidate mean unchanged and deletes the projected action contrast.
    """

    array = np.asarray(values, dtype=np.float64)
    original_shape = array.shape
    flat = array.reshape(array.shape[0], -1)
    directions = np.asarray(basis, dtype=np.float64)
    if directions.ndim != 2 or directions.shape[0] != flat.shape[1]:
        raise ValueError("basis does not match flattened activation width")
    if not np.allclose(
        directions.T @ directions,
        np.eye(directions.shape[1]),
        atol=1e-6,
        rtol=1e-6,
    ):
        raise ValueError("basis columns must be orthonormal")
    residual = candidate_center(flat)
    projected = (residual @ directions) @ directions.T
    return (-float(dose) * projected).reshape(original_shape)


def action_contrast_energy_metrics(baseline, patched):
    """Measure how much candidate-specific output energy survives an edit."""

    base = np.asarray(baseline, dtype=np.float64).reshape(len(baseline), -1)
    edit = np.asarray(patched, dtype=np.float64).reshape(len(patched), -1)
    if base.shape != edit.shape:
        raise ValueError("baseline and patched outputs must have equal shape")
    centered_base = candidate_center(base)
    centered_edit = candidate_center(edit)
    baseline_energy = float(np.sum(centered_base**2))
    patched_energy = float(np.sum(centered_edit**2))
    if baseline_energy <= 1e-12:
        return {
            "baseline_energy": baseline_energy,
            "patched_energy": patched_energy,
            "energy_retention": math.nan,
            "energy_reduction": math.nan,
            "contrast_cosine": math.nan,
        }
    retention = patched_energy / baseline_energy
    cosine_denominator = math.sqrt(baseline_energy * patched_energy)
    cosine = (
        float(np.sum(centered_base * centered_edit) / cosine_denominator)
        if cosine_denominator > 1e-12
        else 0.0
    )
    return {
        "baseline_energy": baseline_energy,
        "patched_energy": patched_energy,
        "energy_retention": float(retention),
        "energy_reduction": float(1.0 - retention),
        "contrast_cosine": cosine,
    }


def physical_diversity_metrics(costs, interaction_counts, tie=1e-4):
    """Return model-blind action-bank eligibility statistics for one state."""

    values = np.asarray(costs, dtype=np.float64)
    contacts = np.asarray(interaction_counts)
    if values.ndim != 1 or contacts.shape != values.shape or len(values) < 2:
        raise ValueError("costs and contact counts must be aligned vectors")
    left, right = np.triu_indices(len(values), k=1)
    margins = np.abs(values[left] - values[right])
    return {
        "cost_min": float(np.min(values)),
        "cost_max": float(np.max(values)),
        "cost_spread": float(np.max(values) - np.min(values)),
        "non_tied_pair_fraction": float(np.mean(margins > float(tie))),
        "contact_branches": int(np.sum(contacts > 0)),
        "total_contacts": int(np.sum(contacts)),
    }





def rotate_vector(vector, angle):
    """Rotate a two-dimensional vector by ``angle`` radians."""

    value = np.asarray(vector, dtype=np.float64)
    if value.shape != (2,):
        raise ValueError("vector must have shape (2,)")
    cosine, sine = np.cos(float(angle)), np.sin(float(angle))
    return np.asarray(
        [cosine * value[0] - sine * value[1],
         sine * value[0] + cosine * value[1]],
        dtype=np.float64,
    )


def unseen_action_bank(toward_block, family, steps=15):
    """Return the preregistered no-op plus twelve antithetic actions.

    The Stage 18 bank used twelve constant directions separated by 30 degrees
    at magnitude 0.12.  Stage 19 holds out either the angular midpoints, two
    new magnitudes, or two new equal-impulse temporal profiles.  Temporal
    profiles have ten active steps at magnitude 0.18, so their vector sum is
    equal to the Stage 18 constant profile (fifteen steps at 0.12).
    """

    direction = np.asarray(toward_block, dtype=np.float64)
    if direction.shape != (2,) or not np.all(np.isfinite(direction)):
        raise ValueError("toward_block must be a finite two-vector")
    norm = float(np.linalg.norm(direction))
    if norm <= 1e-12:
        raise ValueError("toward_block is degenerate")
    direction = direction / norm
    if family not in TRANSFER_FAMILIES:
        raise ValueError(f"unknown transfer family {family!r}")
    if int(steps) != 15:
        raise ValueError("Stage 19 is frozen to fifteen environment steps")

    branches = [np.zeros((steps, 2), dtype=np.float64)]
    for index in range(12):
        phase = 2.0 * np.pi * index / 12.0
        if family == "rotated_direction":
            phase += np.pi / 12.0
        radial = rotate_vector(direction, phase)
        if family == "magnitude_0p08":
            profile = np.full(steps, 0.08, dtype=np.float64)
        elif family == "magnitude_0p16":
            profile = np.full(steps, 0.16, dtype=np.float64)
        elif family == "delayed_equal_impulse":
            profile = np.r_[np.zeros(5), np.full(10, 0.18)]
        elif family == "pulsed_equal_impulse":
            profile = np.r_[np.full(5, 0.18), np.zeros(5), np.full(5, 0.18)]
        else:
            profile = np.full(steps, 0.12, dtype=np.float64)
        branches.append(profile[:, None] * radial[None, :])

    actions = np.stack(branches).astype(np.float32)
    if actions.shape != (13, steps, 2):
        raise RuntimeError(f"bad Stage 19 action-bank shape {actions.shape}")
    for index in range(1, 7):
        if not np.allclose(actions[index], -actions[index + 6], atol=1e-7):
            raise RuntimeError("Stage 19 action bank lost antithetic pairing")
    return actions


def validate_stage18_subspace_arrays(arrays, ambient=102400, max_rank=128):
    """Fail closed if the imported Stage 18 artifact violates its contract."""

    required = {
        "primary_basis",
        "shuffled_basis",
        "channel_square_root",
        "channel_inverse_square_root",
        *(f"random_basis_{draw:02d}" for draw in range(4)),
    }
    missing = sorted(required.difference(arrays))
    if missing:
        raise ValueError(f"Stage 18 artifact is missing arrays: {missing}")
    basis_names = [
        "primary_basis",
        "shuffled_basis",
        *(f"random_basis_{draw:02d}" for draw in range(4)),
    ]
    errors = {}
    for name in basis_names:
        basis = np.asarray(arrays[name], dtype=np.float64)
        if basis.shape != (int(ambient), int(max_rank)):
            raise ValueError(f"{name} has shape {basis.shape}")
        error = float(np.max(np.abs(basis.T @ basis - np.eye(max_rank))))
        if not np.isfinite(error) or error > 1e-10:
            raise ValueError(f"{name} is not orthonormal: {error}")
        errors[name] = error
    for name in ["channel_square_root", "channel_inverse_square_root"]:
        if np.asarray(arrays[name]).shape != (400, 400):
            raise ValueError(f"{name} must have shape (400, 400)")
    return {"validated": True, "orthonormality_max_errors": errors}


def targeted_derangement(size, target, donor, seed):
    """Return a derangement whose target receives the donor's value.

    The remaining indices form one deterministic random cycle.  Consequently
    every candidate changes identity, ``permutation[target] == donor``, and a
    complete interchange makes the target inherit the donor's score.
    """

    size, target, donor = int(size), int(target), int(donor)
    if size < 3 or not (0 <= target < size) or not (0 <= donor < size):
        raise ValueError("invalid targeted-derangement arguments")
    if target == donor:
        raise ValueError("target and donor must differ")
    remaining = [value for value in range(size) if value not in {target, donor}]
    digest = hashlib.sha256(str(seed).encode()).digest()
    rng = np.random.default_rng(int.from_bytes(digest[:8], "big"))
    rng.shuffle(remaining)
    cycle = [target, donor, *remaining]
    permutation = np.empty(size, dtype=np.int64)
    for left, right in zip(cycle, cycle[1:] + cycle[:1]):
        permutation[left] = right
    if permutation[target] != donor or np.any(permutation == np.arange(size)):
        raise RuntimeError("failed to construct targeted derangement")
    return permutation


def stable_action_rank(values, action):
    """Zero-based stable ascending rank of one action (lower is better)."""

    scores = np.asarray(values, dtype=np.float64)
    action = int(action)
    if scores.ndim != 1 or not 0 <= action < len(scores):
        raise ValueError("scores and action are incompatible")
    if not np.all(np.isfinite(scores)):
        raise ValueError("scores contain nonfinite values")
    order = np.argsort(scores, kind="stable")
    return int(np.flatnonzero(order == action)[0])


def select_near_frontier_targets(scores, ranks=(1, 2, 3)):
    """Select fixed baseline rank positions without simulator-outcome access."""

    values = np.asarray(scores, dtype=np.float64)
    if values.ndim != 1 or not np.all(np.isfinite(values)):
        raise ValueError("scores must be a finite vector")
    order = np.argsort(values, kind="stable")
    ranks = tuple(int(value) for value in ranks)
    if len(set(ranks)) != len(ranks) or min(ranks) < 1 or max(ranks) >= len(values):
        raise ValueError("target ranks must be unique non-best positions")
    return int(order[0]), [int(order[value]) for value in ranks]


def planner_steering_metrics(
    baseline_scores,
    patched_scores,
    true_costs,
    permutation,
    target_action,
):
    """Score prediction, ranking, choice, and physical consequences of an edit."""

    baseline = np.asarray(baseline_scores, dtype=np.float64)
    patched = np.asarray(patched_scores, dtype=np.float64)
    physical = np.asarray(true_costs, dtype=np.float64)
    permutation = np.asarray(permutation, dtype=np.int64)
    target_action = int(target_action)
    if not (
        baseline.ndim == 1
        and patched.shape == baseline.shape
        and physical.shape == baseline.shape
        and permutation.shape == baseline.shape
    ):
        raise ValueError("planner metric vectors must be aligned")
    if sorted(permutation.tolist()) != list(range(len(baseline))):
        raise ValueError("permutation is malformed")
    if not all(np.all(np.isfinite(value)) for value in [baseline, patched, physical]):
        raise ValueError("planner metric vectors contain nonfinite values")

    expected = baseline[permutation]
    expected_choice = int(np.argmin(expected))
    baseline_choice = int(np.argmin(baseline))
    patched_choice = int(np.argmin(patched))
    transfer = donor_transfer_metrics(
        baseline[:, None], patched[:, None], permutation
    )
    denominator = float(np.sqrt(np.mean((expected - baseline) ** 2)))
    normalized_error = float(
        np.sqrt(np.mean((patched - expected) ** 2)) / max(denominator, 1e-12)
    )
    baseline_rank = stable_action_rank(baseline, target_action)
    patched_rank = stable_action_rank(patched, target_action)
    return {
        "score_transfer_coefficient": transfer["coefficient"],
        "score_transfer_cosine": transfer["cosine"],
        "score_counterfactual_normalized_rmse": normalized_error,
        "baseline_choice": baseline_choice,
        "expected_counterfactual_choice": expected_choice,
        "patched_choice": patched_choice,
        "target_action": target_action,
        "target_is_expected_choice": bool(expected_choice == target_action),
        "target_rank_baseline": baseline_rank,
        "target_rank_patched": patched_rank,
        "target_rank_gain": int(baseline_rank - patched_rank),
        "target_selected": bool(patched_choice == target_action),
        "choice_matches_counterfactual": bool(patched_choice == expected_choice),
        "choice_flipped": bool(patched_choice != baseline_choice),
        "baseline_selected_true_cost": float(physical[baseline_choice]),
        "patched_selected_true_cost": float(physical[patched_choice]),
        "selected_true_cost_change": float(
            physical[patched_choice] - physical[baseline_choice]
        ),
        "target_true_cost": float(physical[target_action]),
    }

In [ ]:
def to_model_observation(visual, proprio):
    visual_tensor = torch.from_numpy(np.asarray(visual))
    proprio_tensor = torch.from_numpy(np.asarray(proprio))
    if visual_tensor.ndim == 3:
        visual_tensor = visual_tensor[None, None]
        proprio_tensor = proprio_tensor[None, None]
    elif visual_tensor.ndim != 5:
        raise ValueError(f"unexpected visual shape {tuple(visual_tensor.shape)}")
    visual_tensor = visual_tensor.permute(0, 1, 4, 2, 3)
    return {"visual": visual_tensor, "proprio": proprio_tensor}


def configure_repo():
    # Google Drive is reliable for immutable model assets but not as a mutable
    # Git worktree. Use a fresh runtime-local checkout and keep HF/Torch assets
    # in the persistent cache configured by the setup cell.
    repo = Path("/content") / f"stage20-jepa-wms-{REPO_COMMIT[:12]}"
    if repo.exists():
        shutil.rmtree(repo)

    def run_git(command):
        completed = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )
        if completed.returncode != 0:
            raise RuntimeError(
                "git repository setup failed\n"
                f"command: {command!r}\n"
                f"stdout:\n{completed.stdout}\n"
                f"stderr:\n{completed.stderr}"
            )
        return completed

    print(f"Preparing clean ephemeral JEPA-WM source at {repo}")
    run_git(["git", "clone", "--no-checkout", REPO_URL, str(repo)])
    run_git(["git", "-C", str(repo), "fetch", "origin", REPO_COMMIT])
    run_git(["git", "-C", str(repo), "checkout", "--detach", REPO_COMMIT])
    resolved = run_git(
        ["git", "-C", str(repo), "rev-parse", "HEAD"]
    ).stdout.strip()
    if resolved != REPO_COMMIT:
        raise RuntimeError(f"repository pin mismatch: {resolved}")

    # The public hub loader imports a planning entry point with extra evaluation
    # dependencies. Use its equivalent lightweight model constructor.
    hubconf = repo / "hubconf.py"
    text = hubconf.read_text()
    old = "from evals.simu_env_planning.eval import init_module"
    new = (
        "from app.vjepa_wm.modelcustom.simu_env_planning."
        "vit_enc_preds import init_module"
    )
    if old in text:
        hubconf.write_text(text.replace(old, new))
    elif new not in text:
        raise RuntimeError("unexpected hubconf.py at pinned commit")

    # Pin the exact Hugging Face snapshot used by the Stage 7 cache.
    hubconf_text = hubconf.read_text()
    filename_marker = 'filename=f"{model_name}.pth.tar",'
    revision_marker = f'revision="{EXPECTED_HF_REVISION}"'
    if revision_marker not in hubconf_text:
        if hubconf_text.count(filename_marker) != 1:
            raise RuntimeError(
                "cannot pin the checkpoint revision in hubconf.py"
            )
        hubconf_text = hubconf_text.replace(
            filename_marker,
            filename_marker
            + f'\n            revision="{EXPECTED_HF_REVISION}",',
        )
        hubconf.write_text(hubconf_text)
    hubconf_text = hubconf.read_text()
    fallback_marker = (
        "        except Exception:\n"
        "            # Fall back to fbaipublicfiles URL\n"
        "            pass\n"
    )
    fail_closed_marker = "pinned Hugging Face checkpoint retrieval failed"
    if fail_closed_marker not in hubconf_text:
        if hubconf_text.count(fallback_marker) != 1:
            raise RuntimeError(
                "cannot disable mutable checkpoint fallback in hubconf.py"
            )
        fallback_replacement = (
            "        except Exception as error:\n"
            "            raise RuntimeError(\n"
            '                "pinned Hugging Face checkpoint retrieval failed"\n'
            "            ) from error\n"
        )
        hubconf_text = hubconf_text.replace(
            fallback_marker,
            fallback_replacement,
        )
        hubconf.write_text(hubconf_text)

    # PushT and Wall do not use the DROID pose helper.
    model_file = (
        repo / "app/vjepa_wm/modelcustom/simu_env_planning/vit_enc_preds.py"
    )
    text = model_file.read_text()
    old = "from app.plan_common.datasets.droid_dset import compute_new_pose"
    new = (
        "def compute_new_pose(*args, **kwargs):\n"
        "    raise RuntimeError('compute_new_pose is unused in simulator predict_proprio mode')"
    )
    if old in text:
        model_file.write_text(text.replace(old, new))
    elif "compute_new_pose is unused" not in text:
        raise RuntimeError("unexpected vit_enc_preds.py at pinned commit")

    config_paths = [
        repo
        / "configs/evals/simu_env_planning/pt/dino-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/pt/jepa-wm/"
        "pt_L2_cem_sourcedset_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/dino-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
        repo
        / "configs/evals/simu_env_planning/wall/jepa-wm/"
        "wall_L2_cem_sourcerandstate_H6_nas6_ctxt2_r224_alpha0.1_ep96_decode.yaml",
    ]
    for config_path in config_paths:
        config = yaml.safe_load(config_path.read_text())
        heads = config["model_kwargs"]["pretrain_kwargs"]["heads_cfg"]
        heads["architectures"] = {}
        heads["pretrain_dec_path"] = None
        config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    return repo


def make_environment(repo, environment, task=None):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    if environment == "PushT":
        from evals.simu_env_planning.envs.pusht_env.pusht_env import PushTEnv

        return PushTEnv(
            with_velocity=True,
            with_target=True,
            render_size=224,
            relative=True,
            action_scale=100,
        )

    from evals.simu_env_planning.envs.wall_gym_wrap import DEFAULT_CFG
    from evals.simu_env_planning.envs.wall_env.envs.wall import DotWall

    if task is None:
        raise RuntimeError("Stage 14 supports PushT only")
    env = DotWall(
        rng=np.random.default_rng(SEED),
        wall_config=deepcopy(DEFAULT_CFG),
        fix_wall=True,
        cross_wall=False,
        device="cpu",
    )
    env.wall_x = torch.tensor(float(task["wall_x"]))
    env.hole_y = torch.tensor(float(task["door_y"]))
    env.left_wall_x = env.wall_x - env.wall_config.wall_width // 2
    env.right_wall_x = env.wall_x + env.wall_config.wall_width // 2
    return env


def verify_pretrained_assets():
    rows = []
    for name, expected in EXPECTED_PRETRAINED_ASSET_SHA256.items():
        matching = [
            path for path in CACHE_ROOT.rglob(name)
            if sha256_file(path) == expected
        ]
        if not matching:
            raise RuntimeError(f"verified pretrained asset not found: {name}")
        rows.append({"name": name, "path": str(matching[0]), "sha256": expected})
    write_json(OUT / "pretrained_asset_verification.json", rows)
    return rows


def validate_jepa_predictor(model):
    predictor = model.model.predictor
    blocks = list(getattr(predictor, "predictor_blocks", []))
    if predictor.__class__.__name__ != "VisionTransformerAdaLN":
        raise RuntimeError(f"unexpected predictor class {predictor.__class__.__name__}")
    if len(blocks) != 6:
        raise RuntimeError(f"expected six predictor blocks, found {len(blocks)}")
    if not bool(getattr(predictor, "action_encoder_inpred", False)):
        raise RuntimeError("predictor does not internally encode actions")
    if bool(getattr(predictor, "use_activation_checkpointing", False)):
        raise RuntimeError("activation checkpointing must be disabled for hooks/JVPs")
    if int(getattr(model, "ctxt_window", -1)) != 2:
        raise RuntimeError(f"expected context window 2, found {model.ctxt_window}")
    return predictor, blocks


def load_frozen_model():
    model, preprocessor = torch.hub.load(
        str(REPO),
        MODEL_NAME,
        source="local",
        pretrained=True,
        device="cuda:0",
        trust_repo=True,
    )
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    predictor, blocks = validate_jepa_predictor(model)
    verify_pretrained_assets()
    return model, preprocessor, predictor, blocks


def model_action_tensor(preprocessor, selected_actions, horizon):
    selected_actions = np.asarray(selected_actions, dtype=np.float32)
    chunks = torch.from_numpy(
        selected_actions[:, : horizon * FRAMESKIP].reshape(
            ACTIONS_PER_STATE, horizon, FRAMESKIP, 2
        )
    ).float()
    normalized = preprocessor.normalize_actions(chunks)
    return (
        normalized.reshape(ACTIONS_PER_STATE, horizon, -1)
        .permute(1, 0, 2)
        .contiguous()
        .cuda()
    )


def layer_tokens_full(capture):
    if capture.ndim != 3 or capture.shape[1] % 256:
        raise ValueError(f"unexpected block output {tuple(capture.shape)}")
    if capture.shape[-1] != EXPECTED_CARRIER_CHANNELS:
        raise ValueError(
            f"expected carrier width {EXPECTED_CARRIER_CHANNELS}, "
            f"found {capture.shape[-1]}"
        )
    return capture.view(
        capture.shape[0], capture.shape[1] // 256, 256, capture.shape[-1]
    )[:, -1]


def forward_with_carriers(
    initial,
    actions,
    horizon,
    capture_blocks=(),
    intervention=None,
):
    captures = {int(block): [] for block in capture_blocks}
    context = {"step": -1}
    handles = []
    for block_index in capture_blocks:
        def hook(_module, _inputs, output, block_index=int(block_index)):
            captures[block_index].append(output)
            if (
                intervention is None
                or block_index != int(intervention["block"])
                or context["step"] != horizon - 1
            ):
                return output
            value = output.clone()
            view = value.view(
                value.shape[0], value.shape[1] // 256, 256, value.shape[-1]
            )
            replacement = intervention.get("replacement")
            if replacement is not None:
                view[:, -1] = replacement.to(view.device, view.dtype)
            else:
                delta = intervention["delta"].to(view.device, view.dtype)
                view[:, -1] = view[:, -1] + delta
            return view.reshape_as(value)

        handles.append(PREDICTOR_BLOCK_MODULES[block_index].register_forward_hook(hook))

    try:
        batch = actions.shape[1]
        action_batch = actions[:horizon].permute(1, 0, 2).contiguous()
        with torch.inference_mode():
            action_features = MODEL.model.encode_act(action_batch)
            if action_features.shape[-1] != 10:
                raise RuntimeError(
                    f"expected encoded action width 10, found {action_features.shape[-1]}"
                )
            visual_history = initial["visual"].expand(
                batch, *initial["visual"].shape[1:]
            ).detach().clone()
            proprio_history = initial["proprio"].expand(
                batch, *initial["proprio"].shape[1:]
            ).detach().clone()
            predicted_tokens = None
            predicted_proprio = None
            for step_index in range(horizon):
                context["step"] = step_index
                predicted_visual, _, predicted_proprio = MODEL.model.forward_pred(
                    visual_history[:, -MODEL.ctxt_window :],
                    action_features[:, : step_index + 1][:, -MODEL.ctxt_window :],
                    proprio_history[:, -MODEL.ctxt_window :],
                )
                next_visual = predicted_visual[:, -1:]
                next_proprio = predicted_proprio[:, -1:]
                predicted_tokens = next_visual[:, 0, 0].flatten(1, 2)
                if predicted_tokens.shape[1:] != (256, 384):
                    raise RuntimeError(
                        f"expected visual grid [256,384], found {predicted_tokens.shape[1:]}"
                    )
                visual_history = torch.cat([visual_history, next_visual], dim=1)
                proprio_history = torch.cat([proprio_history, next_proprio], dim=1)
        final_captures = {
            block: captures[block][-1] for block in capture_blocks
        }
        return predicted_tokens, predicted_proprio[:, -1], final_captures
    finally:
        for handle in handles:
            handle.remove()


def physical_pose_decoder():
    payload = torch.load(
        ASSET_DIR / "physical_decoders.pt", map_location="cpu", weights_only=False
    )
    projectors = {}

    def decode(tokens):
        outputs = []
        for decoder in payload["decoders"]:
            seed = int(decoder["projection_seed"])
            if seed not in projectors:
                projectors[seed] = CountSketchProjector(
                    tokens.shape[-2] * tokens.shape[-1],
                    int(payload["projection_dim"]),
                    seed,
                )
            features = projectors[seed](tokens)
            mean = torch.as_tensor(
                decoder["mean"], device=tokens.device, dtype=torch.float32
            )
            scale = torch.as_tensor(
                decoder["scale"], device=tokens.device, dtype=torch.float32
            )
            coefficient = torch.as_tensor(
                decoder["coefficient"], device=tokens.device, dtype=torch.float32
            )
            intercept = torch.as_tensor(
                decoder["intercept"], device=tokens.device, dtype=torch.float32
            )
            outputs.append(intercept + ((features - mean) / scale) @ coefficient)
        return torch.stack(outputs).mean(dim=0)

    return decode

In [ ]:
# Freeze two action families and fresh physical state pool before simulator or model data.


def trajectory_specs():
    specs = []
    center = np.asarray([256.0, 256.0])
    total = len(EVALUATION_POOL_TRAJECTORIES)
    for design_index, trajectory_id in enumerate(EVALUATION_POOL_TRAJECTORIES):
        phase = 0.29 + 2.0 * np.pi * design_index / total
        block = center + 44.0 * np.asarray([np.cos(phase), np.sin(phase)])
        block_angle = ((2.1 * phase + np.pi) % (2.0 * np.pi)) - np.pi
        approach = phase + [np.pi / 3, 2 * np.pi / 3, 4 * np.pi / 3, 5 * np.pi / 3][design_index % 4]
        approach += 0.11 * np.sin(2 * design_index)
        agent = block + APPROACH_DISTANCE * np.asarray([np.cos(approach), np.sin(approach)])
        goal_index = (17 * design_index + 5) % total
        goal_phase = 0.71 + 2.0 * np.pi * goal_index / total
        goal_xy = center + 73.0 * np.asarray([np.cos(goal_phase), np.sin(goal_phase)])
        common = {
            "design_index": int(design_index),
            "trajectory_id": int(trajectory_id),
            "time_index": 0,
            "physical_step": 0,
            "split": "evaluation",
            "evaluation_seed": int(DESIGN_SEED + 1013 * design_index),
            "goal": np.asarray(
                [goal_xy[0], goal_xy[1], ((1.2 * goal_phase + np.pi) % (2.0 * np.pi)) - np.pi],
                dtype=np.float64,
            ),
            "state": np.asarray(
                [agent[0], agent[1], block[0], block[1], block_angle, 0.0, 0.0, 0.0, 0.0, 0.0],
                dtype=np.float64,
            ),
        }
        for family_index, family in enumerate(TRANSFER_FAMILIES):
            specs.append(
                {
                    **common,
                    "record_id": int(600000 + 1000 * family_index + trajectory_id),
                    "task_id": int(TASK_ID_OFFSET + design_index),
                    "action_family": family,
                    "family_index": int(family_index),
                }
            )
    return specs


ALL_POOL_SPECS = trajectory_specs()
POOL_SPECS = [
    row for row in ALL_POOL_SPECS
    if row["trajectory_id"] in ACTIVE_EVALUATION_POOL_TRAJECTORIES
]


def candidate_action_bank(record):
    state = np.asarray(record["state"], dtype=np.float64)
    if state.shape != (10,):
        raise ValueError("candidate state must be a ten-dimensional dynamic PushT state")
    return unseen_action_bank(state[2:4] - state[:2], record["action_family"], ACTION_STEPS)


np.savez_compressed(
    DESIGN_DIR / "stage20_steering_pool_design.npz",
    record_ids=np.asarray([row["record_id"] for row in ALL_POOL_SPECS]),
    trajectory_ids=np.asarray([row["trajectory_id"] for row in ALL_POOL_SPECS]),
    action_families=np.asarray([row["action_family"] for row in ALL_POOL_SPECS]),
    initial_states=np.stack([row["state"] for row in ALL_POOL_SPECS]),
    goals=np.stack([row["goal"] for row in ALL_POOL_SPECS]),
)
POOL_MANIFEST = {
    "specs": [
        {
            **{key: value for key, value in row.items() if key not in {"state", "goal"}},
            "state": row["state"].tolist(),
            "goal": row["goal"].tolist(),
        }
        for row in ALL_POOL_SPECS
    ],
    "active_pool_trajectory_ids": ACTIVE_EVALUATION_POOL_TRAJECTORIES,
    "action_families": TRANSFER_FAMILIES,
    "target_per_family": ACTIVE_EVALUATION_TARGET_PER_FAMILY,
    "target_baseline_ranks": ACTIVE_TARGET_BASELINE_RANKS,
    "physical_selection_uses_model_outputs": False,
    "steering_target_selection_uses_baseline_model_scores_only": True,
    "steering_target_selection_uses_simulator_costs": False,
    "eligibility": {
        "min_cost_spread": MIN_ELIGIBLE_COST_SPREAD,
        "min_non_tied_pair_fraction": MIN_ELIGIBLE_NON_TIED_PAIR_FRACTION,
        "min_contact_branches": MIN_ELIGIBLE_CONTACT_BRANCHES,
        "tie": PHYSICAL_COST_TIE,
    },
}
write_json(DESIGN_DIR / "candidate_pool_manifest.json", POOL_MANIFEST)
DESIGN_FREEZE = {
    "created_before_simulator_or_model_data": True,
    "protocol_id": PROTOCOL_ID,
    "run_signature": RUN_SIGNATURE,
    "source_identity": SOURCE_IDENTITY,
    "candidate_pool_sha256": sha256_file(DESIGN_DIR / "stage20_steering_pool_design.npz"),
    "pool_manifest_sha256": sha256_file(DESIGN_DIR / "candidate_pool_manifest.json"),
    "expected_stage18_subspace_sha256": EXPECTED_STAGE18_SUBSPACE_SHA256,
    "expected_stage19_decision_sha256": EXPECTED_STAGE19_DECISION_SHA256,
    "fixed_block": FIXED_BLOCK,
    "primary_steering_rank": PRIMARY_STEERING_RANK,
    "sensitivity_rank": SENSITIVITY_RANK,
    "subspace_refit_allowed": False,
    "visual_evaluation_used": False,
    "model_loaded": bool("MODEL" in globals()),
}
if DESIGN_FREEZE["model_loaded"]:
    raise RuntimeError("model was loaded before Stage 20 design freeze")
write_json(DESIGN_DIR / "design_freeze.json", DESIGN_FREEZE)

In [ ]:
# Generate and select physical truth before loading any model or encoder.


def record_task(record):
    return {"goal": np.asarray(record["goal"], dtype=np.float64).tolist()}


def dynamic_state_from_environment(environment):
    return np.asarray(
        [
            *environment.agent.position,
            *environment.block.position,
            float(environment.block.angle),
            *environment.agent.velocity,
            *environment.block.velocity,
            float(environment.block.angular_velocity),
        ],
        dtype=np.float64,
    )


def reset_dynamic_environment(dynamic_state, task, seed):
    state = np.asarray(dynamic_state, dtype=np.float64)
    if state.shape != (10,):
        raise ValueError(f"expected ten-dimensional dynamic state, found {state.shape}")
    environment = make_environment(REPO, ENVIRONMENT)
    environment.seed(int(seed))
    environment.reset_to_state = np.asarray([*state[:5], 0.0, 0.0], dtype=np.float64)
    environment.reset()
    environment.agent.position = tuple(state[:2])
    environment.block.angle = float(state[4])
    environment.block.position = tuple(state[2:4])
    environment.agent.velocity = tuple(state[5:7])
    environment.block.velocity = tuple(state[7:9])
    environment.block.angular_velocity = float(state[9])
    environment.set_task_goal(np.asarray(task["goal"], dtype=np.float64))
    restored = dynamic_state_from_environment(environment)
    if not np.allclose(restored, state, atol=1e-12, rtol=0):
        raise RuntimeError(f"full dynamic restoration drifted: {np.max(np.abs(restored - state))}")
    observation = {
        "visual": np.asarray(environment.render("rgb_array")).copy(),
        "proprio": np.asarray([*environment.agent.position, *environment.agent.velocity], dtype=np.float32),
    }
    return environment, observation


def rollout_dynamic_branch(record, actions):
    environment, initial = reset_dynamic_environment(
        record["state"], record_task(record), record["evaluation_seed"]
    )
    cumulative = 0
    endpoint_observation = None
    endpoint_state = None
    try:
        for step, action in enumerate(actions, start=1):
            observation, _, _, info = environment.step(action)
            cumulative += int(info.get("n_contacts", 0))
            if step == ACTION_STEPS:
                endpoint_observation = {
                    "visual": np.asarray(observation["visual"]).copy(),
                    "proprio": np.asarray(observation["proprio"]).copy(),
                }
                endpoint_state = dynamic_state_from_environment(environment)
    finally:
        environment.close()
    if endpoint_observation is None or endpoint_state is None:
        raise RuntimeError("dynamic rollout missed the primary horizon")
    return initial, endpoint_observation, endpoint_state, cumulative


def exact_dynamic_restore_test(record):
    first, first_observation = reset_dynamic_environment(
        record["state"], record_task(record), record["evaluation_seed"]
    )
    second, second_observation = reset_dynamic_environment(
        record["state"], record_task(record), record["evaluation_seed"]
    )
    first_state = dynamic_state_from_environment(first)
    second_state = dynamic_state_from_environment(second)
    test_action = candidate_action_bank(record)[1, 0]
    first.step(test_action)
    second.step(test_action)
    first_next = dynamic_state_from_environment(first)
    second_next = dynamic_state_from_environment(second)
    first.close()
    second.close()
    result = {
        "state_exact": bool(np.allclose(first_state, second_state, atol=1e-12, rtol=0)),
        "visual_exact": bool(np.array_equal(first_observation["visual"], second_observation["visual"])),
        "proprio_exact": bool(np.array_equal(first_observation["proprio"], second_observation["proprio"])),
        "one_step_continuation_exact": bool(np.allclose(first_next, second_next, atol=1e-12, rtol=0)),
    }
    result["passed"] = bool(all(result.values()))
    if not result["passed"]:
        raise RuntimeError(f"full dynamic restore test failed: {result}")
    return result


def branch_path(record_id):
    return TRUTH_DIR / f"state_{int(record_id):04d}.npz"


def generate_truth(records, progress_name):
    started = time.perf_counter()
    for index, record in enumerate(records):
        destination = branch_path(record["record_id"])
        if destination.exists():
            PROVENANCE_COUNTS["cache_hits"] += 1
            raise RuntimeError(f"fresh-run truth shard already exists: {destination}")
        action_bank = candidate_action_bank(record)
        initials, initial_proprios = [], []
        endpoint_visuals, endpoint_states, interaction_counts = [], [], []
        for action in action_bank:
            initial, endpoint, state, contacts = rollout_dynamic_branch(record, action)
            initials.append(initial["visual"])
            initial_proprios.append(initial["proprio"])
            endpoint_visuals.append(endpoint["visual"])
            endpoint_states.append(state)
            interaction_counts.append(contacts)
        if not all(np.array_equal(initials[0], value) for value in initials[1:]):
            raise AssertionError("initial visual drift across candidate branches")
        if not all(np.array_equal(initial_proprios[0], value) for value in initial_proprios[1:]):
            raise AssertionError("initial proprio drift across candidate branches")
        atomic_npz(
            destination,
            record_id=np.asarray(record["record_id"], dtype=np.int64),
            trajectory_id=np.asarray(record["trajectory_id"], dtype=np.int64),
            task_id=np.asarray(record["task_id"], dtype=np.int64),
            split=np.asarray(record["split"]),
            action_family=np.asarray(record["action_family"]),
            state=np.asarray(record["state"], dtype=np.float64),
            goal=np.asarray(record["goal"], dtype=np.float64),
            initial_visual=np.asarray(initials[0], dtype=np.uint8),
            initial_proprio=np.asarray(initial_proprios[0], dtype=np.float32),
            selected_actions=action_bank.astype(np.float32),
            endpoint_visuals=np.asarray(endpoint_visuals, dtype=np.uint8),
            endpoint_states=np.asarray(endpoint_states, dtype=np.float64),
            interaction_counts=np.asarray(interaction_counts, dtype=np.int32),
        )
        PROVENANCE_COUNTS["truth_generated"] += 1
        write_json(
            OUT / f"{progress_name}_progress.json",
            {"completed": index + 1, "total": len(records), "last_record_id": int(record["record_id"])},
        )
    TIMINGS[f"{progress_name}_seconds"] = time.perf_counter() - started


def truth_eligibility(record):
    with np.load(branch_path(record["record_id"])) as payload:
        endpoints = payload["endpoint_states"].astype(np.float64)
        contacts = payload["interaction_counts"].astype(np.int64)
        actions = payload["selected_actions"]
        initial_visual = payload["initial_visual"]
    costs = decoded_task_cost(pose_target(endpoints), np.asarray(record["goal"], dtype=np.float64))
    metrics = physical_diversity_metrics(costs, contacts, tie=PHYSICAL_COST_TIE)
    eligible = bool(
        metrics["cost_spread"] >= MIN_ELIGIBLE_COST_SPREAD
        and metrics["non_tied_pair_fraction"] >= MIN_ELIGIBLE_NON_TIED_PAIR_FRACTION
        and metrics["contact_branches"] >= MIN_ELIGIBLE_CONTACT_BRANCHES
    )
    return {
        "record_id": int(record["record_id"]),
        "trajectory_id": int(record["trajectory_id"]),
        "task_id": int(record["task_id"]),
        "split": record["split"],
        "action_family": record["action_family"],
        **metrics,
        "eligible": eligible,
        "action_sha256": array_sha256(actions),
        "endpoint_state_sha256": array_sha256(endpoints),
        "initial_visual_sha256": array_sha256(initial_visual),
    }


def select_records(records, target):
    rows = [truth_eligibility(record) for record in records]
    selected_ids = [row["record_id"] for row in rows if row["eligible"]][: int(target)]
    if len(selected_ids) != int(target):
        raise RuntimeError(
            f"physical eligibility produced {len(selected_ids)} records but requires {target}"
        )
    chosen = [record for record in records if record["record_id"] in selected_ids]
    return chosen, rows


def freeze_maps_by_family(records_by_family):
    permutations = {}
    wrong = {}
    for family, records in records_by_family.items():
        identifiers = sorted(int(record["record_id"]) for record in records)
        if len(identifiers) < 2:
            raise RuntimeError(f"{family} needs at least two records for wrong-state control")
        for index, record_id in enumerate(identifiers):
            permutations[str(record_id)] = fixed_derangement(
                ACTIONS_PER_STATE,
                stable_seed(PERMUTATION_SEED, record_id, family, "donor"),
            ).tolist()
            wrong[str(record_id)] = identifiers[(index + 1) % len(identifiers)]
    return permutations, wrong




if not PIPELINE_FAILED:
    try:
        REPO = configure_repo()
        RESTORE_TEST = exact_dynamic_restore_test(POOL_SPECS[0])
        write_json(OUT / "restore_test.json", RESTORE_TEST)
        generate_truth(POOL_SPECS, "truth_unseen_action_pool")
        if "MODEL" in globals():
            raise RuntimeError("model was loaded before physical eligibility selection")
        FAMILY_RECORDS = {}
        FAMILY_ELIGIBILITY_ROWS = {}
        for family in TRANSFER_FAMILIES:
            family_pool = [row for row in POOL_SPECS if row["action_family"] == family]
            chosen, rows = select_records(
                family_pool, ACTIVE_EVALUATION_TARGET_PER_FAMILY
            )
            FAMILY_RECORDS[family] = chosen
            FAMILY_ELIGIBILITY_ROWS[family] = rows
        ALL_EVALUATION_RECORDS = [
            record for family in TRANSFER_FAMILIES for record in FAMILY_RECORDS[family]
        ]
        ACTIVE_EVALUATION_TRAJECTORIES_BY_FAMILY = {
            family: [int(row["trajectory_id"]) for row in FAMILY_RECORDS[family]]
            for family in TRANSFER_FAMILIES
        }
        donor_permutations, wrong_state_map = freeze_maps_by_family(FAMILY_RECORDS)
        all_eligibility = [
            row for family in TRANSFER_FAMILIES for row in FAMILY_ELIGIBILITY_ROWS[family]
        ]
        write_csv(EVIDENCE_DIR / "physical_eligibility_rows.csv", all_eligibility)
        SELECTION_CERTIFICATE = {
            "selection_completed_before_model_load": True,
            "selection_used_only_simulator_truth": True,
            "selected_trajectory_ids_by_family": ACTIVE_EVALUATION_TRAJECTORIES_BY_FAMILY,
            "eligible_pool_count_by_family": {
                family: int(sum(row["eligible"] for row in FAMILY_ELIGIBILITY_ROWS[family]))
                for family in TRANSFER_FAMILIES
            },
            "donor_permutations": donor_permutations,
            "wrong_state_map": wrong_state_map,
            "wrong_state_within_action_family": True,
            "eligibility_rows_sha256": sha256_file(EVIDENCE_DIR / "physical_eligibility_rows.csv"),
        }
        write_json(DESIGN_DIR / "physical_selection_freeze.json", SELECTION_CERTIFICATE)
        memory_report("physical_truth_and_selection_complete")
    except Exception:
        record_failure("physical_truth_selection")

In [ ]:
# Bind successful Stages 18/19 and load the exact frozen subspaces before model activations.
PRIOR_ARTIFACTS_VALIDATED = False
if not PIPELINE_FAILED:
    try:
        verify_executed_notebook_through(
            "# Bind successful Stages 18/19 and load the exact frozen subspaces before model activations."
        )
        frozen_subspace_path = Path(STAGE18_SUBSPACE_PATH)
        if not frozen_subspace_path.is_file():
            raise FileNotFoundError(f"Stage 18 raw subspace is missing: {frozen_subspace_path}")
        observed_subspace_sha256 = sha256_file(frozen_subspace_path)
        if observed_subspace_sha256 != EXPECTED_STAGE18_SUBSPACE_SHA256:
            raise RuntimeError(
                f"Stage 18 subspace hash mismatch: {observed_subspace_sha256}"
            )
        stage18_run_dir = frozen_subspace_path.parent.parent
        stage18_decision_path = stage18_run_dir / "stage18_decision.json"
        stage18_manifest_path = stage18_run_dir / "subspaces/subspace_manifest.json"
        stage18_source_path = stage18_run_dir / "source_identity.json"
        for path in [stage18_decision_path, stage18_manifest_path, stage18_source_path]:
            if not path.is_file():
                raise FileNotFoundError(f"Stage 18 provenance file is missing: {path}")
        stage18_decision = json.loads(stage18_decision_path.read_text())
        stage18_manifest = json.loads(stage18_manifest_path.read_text())
        stage18_source = json.loads(stage18_source_path.read_text())
        if stage18_decision.get("status") != EXPECTED_STAGE18_STATUS:
            raise RuntimeError("Stage 18 decision is not the successful confirmation")
        if not bool(stage18_decision.get("confirmation_eligible", False)):
            raise RuntimeError("Stage 18 decision was not claim eligible")
        if stage18_manifest.get("subspace_sha256") != EXPECTED_STAGE18_SUBSPACE_SHA256:
            raise RuntimeError("Stage 18 manifest does not bind the required subspace")
        if stage18_source.get("resolved_commit") != EXPECTED_STAGE18_SOURCE_COMMIT:
            raise RuntimeError("Stage 18 source commit mismatch")
        with np.load(frozen_subspace_path) as payload:
            FROZEN_SUBSPACES = {name: payload[name].copy() for name in payload.files}
        artifact_contract = validate_stage18_subspace_arrays(
            FROZEN_SUBSPACES,
            ambient=EXPECTED_STAGE18_AMBIENT_DIMENSION,
            max_rank=EXPECTED_STAGE18_MAX_RANK,
        )

        stage19_decision_path = Path(STAGE19_DECISION_PATH)
        if not stage19_decision_path.is_file():
            raise FileNotFoundError(f"Stage 19 decision is missing: {stage19_decision_path}")
        if sha256_file(stage19_decision_path) != EXPECTED_STAGE19_DECISION_SHA256:
            raise RuntimeError("Stage 19 decision hash mismatch")
        stage19_source_path = stage19_decision_path.parent / "source_identity.json"
        if not stage19_source_path.is_file():
            raise FileNotFoundError(f"Stage 19 source identity is missing: {stage19_source_path}")
        if sha256_file(stage19_source_path) != EXPECTED_STAGE19_SOURCE_IDENTITY_SHA256:
            raise RuntimeError("Stage 19 source-identity hash mismatch")
        stage19_decision = json.loads(stage19_decision_path.read_text())
        stage19_source = json.loads(stage19_source_path.read_text())
        if stage19_decision.get("status") != EXPECTED_STAGE19_STATUS:
            raise RuntimeError("Stage 19 did not confirm all unseen action families")
        if not bool(stage19_decision.get("confirmation_eligible", False)):
            raise RuntimeError("Stage 19 decision was not claim eligible")
        if stage19_source.get("resolved_commit") != EXPECTED_STAGE19_SOURCE_COMMIT:
            raise RuntimeError("Stage 19 source commit mismatch")
        if not all(
            family in stage19_decision.get("passed_action_families", [])
            for family in TRANSFER_FAMILIES
        ):
            raise RuntimeError("Stage 19 did not pass both Stage 20 action families")

        PRIOR_ARTIFACT_CERTIFICATE = {
            "validated_before_stage20_model_activations": True,
            "stage18_subspace_path": str(frozen_subspace_path),
            "stage18_subspace_bytes": int(frozen_subspace_path.stat().st_size),
            "stage18_subspace_sha256": observed_subspace_sha256,
            "stage18_decision_status": stage18_decision["status"],
            "stage18_artifact_contract": artifact_contract,
            "stage19_decision_path": str(stage19_decision_path),
            "stage19_decision_sha256": sha256_file(stage19_decision_path),
            "stage19_source_identity_sha256": sha256_file(stage19_source_path),
            "stage19_decision_status": stage19_decision["status"],
            "stage19_passed_required_families": TRANSFER_FAMILIES,
            "stage20_subspace_refit": False,
            "stage20_basis_tuning": False,
        }
        write_json(OUT / "prior_artifact_certificate.json", PRIOR_ARTIFACT_CERTIFICATE)
        PRIOR_ARTIFACTS_VALIDATED = True
        memory_report("prior_artifacts_validated")
    except Exception:
        record_failure("prior_artifact_import")

In [ ]:
# Load frozen JEPA-WM, generate baselines, and freeze near-frontier steering targets.


def state_model_inputs(record_id, horizon=PRIMARY_HORIZON):
    with np.load(branch_path(record_id)) as truth:
        initial_visual = truth["initial_visual"]
        initial_proprio = truth["initial_proprio"]
        selected_actions = truth["selected_actions"]
    with torch.inference_mode():
        initial = MODEL.encode(to_model_observation(initial_visual, initial_proprio))
    initial = {name: value.detach() for name, value in initial.items()}
    actions = model_action_tensor(PREPROCESSOR, selected_actions, horizon)
    return initial, actions


def baseline_path(record_id):
    return BASELINE_DIR / f"state_{int(record_id):04d}.npz"


def load_baseline(record_id):
    with np.load(baseline_path(record_id)) as payload:
        return {name: payload[name].copy() for name in payload.files}


def extract_baselines(records, blocks):
    started = time.perf_counter()
    for index, record in enumerate(records):
        destination = baseline_path(record["record_id"])
        if destination.exists():
            PROVENANCE_COUNTS["cache_hits"] += 1
            raise RuntimeError(f"fresh-run baseline shard already exists: {destination}")
        initial, actions = state_model_inputs(record["record_id"])
        with torch.inference_mode():
            predicted, predicted_proprio, captures = forward_with_carriers(
                initial,
                actions,
                PRIMARY_HORIZON,
                capture_blocks=blocks,
            )
            train_sketch = TRAIN_OUTPUT_PROJECTOR(predicted).cpu().numpy()
            eval_sketch = EVAL_OUTPUT_PROJECTOR(predicted).cpu().numpy()
            decoded_pose = DECODE_PHYSICAL_POSE(predicted).cpu().numpy()
        carriers = np.stack(
            [
                layer_tokens_full(captures[block]).detach().float().cpu().numpy()
                for block in blocks
            ]
        )
        atomic_npz(
            destination,
            record_id=np.asarray(record["record_id"], dtype=np.int64),
            trajectory_id=np.asarray(record["trajectory_id"], dtype=np.int64),
            time_index=np.asarray(record["time_index"], dtype=np.int64),
            blocks=np.asarray(blocks, dtype=np.int64),
            # Float32 is deliberate: the full-swap positive control must use
            # the exact cached carrier rather than a quantized approximation.
            carriers=carriers.astype(np.float32),
            output_train_sketch=train_sketch.astype(np.float32),
            output_eval_sketch=eval_sketch.astype(np.float32),
            decoded_pose=decoded_pose.astype(np.float32),
            predicted_proprio=predicted_proprio.detach().float().cpu().numpy(),
        )
        PROVENANCE_COUNTS["baseline_generated"] += 1
        write_json(
            OUT / f"baseline_{record['split']}_progress.json",
            {"completed": index + 1, "total": len(records), "last_record_id": int(record["record_id"])},
        )
        del initial, actions, predicted, predicted_proprio, captures, carriers
        gc.collect()
        torch.cuda.empty_cache()
    TIMINGS[f"baseline_{records[0]['split']}_seconds"] = time.perf_counter() - started


def carrier_for_block(payload, block):
    blocks = payload["blocks"].astype(int).tolist()
    if int(block) not in blocks:
        raise RuntimeError(f"block {block} is absent from baseline shard")
    return payload["carriers"][blocks.index(int(block))].astype(np.float32)


def hook_identity_test(record_id):
    initial, actions = state_model_inputs(record_id)
    block = int(ACTIVE_BLOCKS[0])
    with torch.inference_mode():
        baseline, _, _ = forward_with_carriers(
            initial, actions, PRIMARY_HORIZON, capture_blocks=[block]
        )
        patched, _, _ = forward_with_carriers(
            initial,
            actions,
            PRIMARY_HORIZON,
            capture_blocks=[block],
            intervention={
                "block": block,
                "delta": torch.zeros(
                    ACTIONS_PER_STATE,
                    256,
                    EXPECTED_CARRIER_CHANNELS,
                    device="cuda",
                    dtype=torch.float32,
                ),
            },
        )
    error = float(torch.max(torch.abs(patched - baseline)).cpu())
    result = {"record_id": int(record_id), "block": block, "max_abs_error": error, "passed": error <= MAX_ZERO_EDIT_ERROR}
    if not result["passed"]:
        raise RuntimeError(f"zero intervention changed the model output: {result}")
    write_json(OUT / "hook_identity_test.json", result)
    return result


def forward_benchmark(record_id):
    initial, actions = state_model_inputs(record_id)
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        _, _, _ = forward_with_carriers(
            initial, actions, PRIMARY_HORIZON, capture_blocks=[int(ACTIVE_BLOCKS[0])]
        )
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    interventions_per_record = ACTIVE_INTERVENTION_FORWARDS_PER_RECORD
    total_eval_records = len(ALL_EVALUATION_RECORDS)
    estimate = seconds * interventions_per_record * total_eval_records / 60.0
    result = {
        "seconds_per_candidate_batch": seconds,
        "intervention_forwards_per_record": interventions_per_record,
        "evaluation_records": total_eval_records,
        "estimated_intervention_minutes": estimate,
        "warning_threshold_minutes": MAX_ESTIMATED_TOTAL_MINUTES,
    }
    write_json(OUT / "forward_benchmark.json", result)
    if estimate > MAX_ESTIMATED_TOTAL_MINUTES and not CONTINUE_AFTER_BENCHMARK:
        raise RuntimeError(
            "measured estimate exceeds the configured credit guard; set "
            "CONTINUE_AFTER_BENCHMARK=True only after reviewing forward_benchmark.json"
        )
    return result




def freeze_steering_targets(records):
    target_map = {}
    rows = []
    for record in records:
        record_id = int(record["record_id"])
        payload = load_baseline(record_id)
        goal = np.asarray(record["goal"], dtype=np.float64)
        baseline_scores = decoded_task_cost(
            payload["decoded_pose"].astype(np.float64), goal
        )
        donor, targets = select_near_frontier_targets(
            baseline_scores, ACTIVE_TARGET_BASELINE_RANKS
        )
        entries = []
        for slot, (baseline_rank, target) in enumerate(
            zip(ACTIVE_TARGET_BASELINE_RANKS, targets)
        ):
            permutation = targeted_derangement(
                ACTIONS_PER_STATE,
                target,
                donor,
                stable_seed(PERMUTATION_SEED, record_id, slot, "planner_target"),
            )
            expected_scores = baseline_scores[permutation]
            expected_choice = int(np.argmin(expected_scores))
            if expected_choice != int(target):
                raise RuntimeError("complete targeted interchange does not select target")
            entry = {
                "target_slot": int(slot),
                "target_baseline_rank": int(baseline_rank),
                "target_action": int(target),
                "donor_action": int(donor),
                "permutation": permutation.tolist(),
                "expected_counterfactual_choice": expected_choice,
                "baseline_score_sha256": array_sha256(baseline_scores),
            }
            entries.append(entry)
            rows.append(
                {
                    "record_id": record_id,
                    "trajectory_id": int(record["trajectory_id"]),
                    "action_family": record["action_family"],
                    **{key: value for key, value in entry.items() if key != "permutation"},
                    "permutation": " ".join(str(value) for value in permutation),
                }
            )
        target_map[str(record_id)] = entries
    write_csv(DESIGN_DIR / "steering_target_rows.csv", rows)
    freeze = {
        "created_after_baseline_predictions_before_any_intervention": True,
        "selection_rule": "baseline planner ranks 2, 3, and 4; smoke uses rank 2 only",
        "uses_baseline_model_scores": True,
        "uses_simulator_endpoint_costs": False,
        "uses_intervention_outputs": False,
        "target_map": target_map,
        "target_rows_sha256": sha256_file(DESIGN_DIR / "steering_target_rows.csv"),
        "prior_artifact_certificate_sha256": sha256_file(
            OUT / "prior_artifact_certificate.json"
        ),
    }
    write_json(DESIGN_DIR / "steering_target_freeze.json", freeze)
    return target_map, freeze


EVALUATION_OPENED = False
if not PIPELINE_FAILED:
    try:
        if not PRIOR_ARTIFACTS_VALIDATED:
            raise RuntimeError("successful prior artifacts must be validated before model loading")
        MODEL, PREPROCESSOR, PREDICTOR, PREDICTOR_BLOCK_MODULES = load_frozen_model()
        if len(PREDICTOR_BLOCK_MODULES) != 6:
            raise RuntimeError("predictor block count changed")
        for module in PREDICTOR_BLOCK_MODULES:
            if not isinstance(module, torch.nn.Module) or getattr(module, "register_forward_hook", None) is None:
                raise RuntimeError("predictor block does not support forward hooks")
        TRAIN_OUTPUT_PROJECTOR = CountSketchProjector(
            256 * 384, OUTPUT_SKETCH_DIM, TRAIN_OUTPUT_SKETCH_SEED
        )
        EVAL_OUTPUT_PROJECTOR = CountSketchProjector(
            256 * 384, OUTPUT_SKETCH_DIM, EVAL_OUTPUT_SKETCH_SEED
        )
        DECODE_PHYSICAL_POSE = physical_pose_decoder()
        first_record_id = ALL_EVALUATION_RECORDS[0]["record_id"]
        HOOK_IDENTITY = hook_identity_test(first_record_id)
        FORWARD_BENCHMARK = forward_benchmark(first_record_id)
        extract_baselines(ALL_EVALUATION_RECORDS, [FIXED_BLOCK])
        STEERING_TARGETS, STEERING_TARGET_FREEZE = freeze_steering_targets(
            ALL_EVALUATION_RECORDS
        )
        EVALUATION_OPENED = True
        write_json(
            OUT / "evaluation_open_certificate.json",
            {
                "opened": True,
                "source_identity": SOURCE_IDENTITY,
                "prior_artifact_certificate_sha256": sha256_file(
                    OUT / "prior_artifact_certificate.json"
                ),
                "physical_selection_freeze_sha256": sha256_file(
                    DESIGN_DIR / "physical_selection_freeze.json"
                ),
                "steering_target_freeze_sha256": sha256_file(
                    DESIGN_DIR / "steering_target_freeze.json"
                ),
                "records_by_family": {
                    family: len(FAMILY_RECORDS[family]) for family in TRANSFER_FAMILIES
                },
                "targets_per_record": len(ACTIVE_TARGET_BASELINE_RANKS),
                "intervention_outputs_seen_during_target_selection": [],
                "simulator_endpoint_costs_seen_during_target_selection": [],
            },
        )
        memory_report("baselines_and_steering_targets_frozen")
    except Exception:
        record_failure("steering_model_baselines_and_targets")

In [ ]:
# Intervene on the frozen subspaces and measure prediction, ranking, choice, and physical cost.


def load_frozen_subspaces():
    if not PRIOR_ARTIFACTS_VALIDATED:
        raise RuntimeError("prior artifacts are not validated")
    return FROZEN_SUBSPACES


def whiten_carrier(values, subspaces):
    return transform_primal_channels(
        np.asarray(values, dtype=np.float64),
        subspaces["channel_inverse_square_root"],
    )


def native_edit(values, subspaces):
    return inverse_transform_primal_channels(
        np.asarray(values, dtype=np.float64), subspaces["channel_square_root"]
    )


def truth_costs(record):
    with np.load(branch_path(record["record_id"])) as truth:
        endpoints = truth["endpoint_states"].astype(np.float64)
        goal = truth["goal"].astype(np.float64)
    return decoded_task_cost(pose_target(endpoints), goal), goal


def intervention_path(record_id):
    return INTERVENTION_DIR / f"state_{int(record_id):06d}.json"


def finite_json_rows(rows):
    return [
        {
            key: None
            if isinstance(value, (float, np.floating)) and not np.isfinite(value)
            else value
            for key, value in row.items()
        }
        for row in rows
    ]


def wrong_state_delta(current_white, wrong_white, permutation, basis):
    current_residual = candidate_center(current_white.reshape(ACTIONS_PER_STATE, -1))
    wrong_residual = candidate_center(wrong_white.reshape(ACTIONS_PER_STATE, -1))
    difference = wrong_residual[permutation] - current_residual
    return ((difference @ basis) @ basis.T).reshape(current_white.shape)


def make_result_row(
    record,
    target_entry,
    condition,
    control_family,
    mode,
    rank,
    dose,
    baseline_output,
    patched_output,
    baseline_pose,
    patched_pose,
    true_cost,
    goal,
    edit_norm,
    primary_swap_norm,
    full_swap_norm,
):
    permutation = np.asarray(target_entry["permutation"], dtype=np.int64)
    target_action = int(target_entry["target_action"])
    baseline_scores = decoded_task_cost(baseline_pose, goal)
    patched_scores = decoded_task_cost(patched_pose, goal)
    output = donor_transfer_metrics(baseline_output, patched_output, permutation)
    pose = donor_transfer_metrics(baseline_pose, patched_pose, permutation)
    energy = action_contrast_energy_metrics(baseline_output, patched_output)
    steering = planner_steering_metrics(
        baseline_scores,
        patched_scores,
        true_cost,
        permutation,
        target_action,
    )
    return {
        "record_id": int(record["record_id"]),
        "trajectory_id": int(record["trajectory_id"]),
        "task_id": int(record["task_id"]),
        "action_family": record["action_family"],
        "target_slot": int(target_entry["target_slot"]),
        "target_baseline_rank_frozen": int(target_entry["target_baseline_rank"]),
        "donor_action_frozen": int(target_entry["donor_action"]),
        "selected_block": FIXED_BLOCK,
        "condition": condition,
        "control_family": control_family,
        "mode": mode,
        "rank": int(rank),
        "dose": float(dose),
        "output_coefficient": output["coefficient"],
        "output_cosine": output["cosine"],
        "output_mean_shift_ratio": output["mean_shift_ratio"],
        "pose_coefficient": pose["coefficient"],
        "pose_cosine": pose["cosine"],
        "output_contrast_energy_reduction": energy["energy_reduction"],
        "output_contrast_cosine": energy["contrast_cosine"],
        **steering,
        "carrier_edit_whitened_norm": float(edit_norm),
        "primary_swap_norm": float(primary_swap_norm),
        "full_swap_norm": float(full_swap_norm),
        "edit_to_full_swap_ratio": float(edit_norm) / max(float(full_swap_norm), 1e-12),
    }


def intervention_specs(record, carrier, subspaces):
    record_id = int(record["record_id"])
    white = whiten_carrier(carrier, subspaces)
    primary_basis = subspaces["primary_basis"][:, :PRIMARY_STEERING_RANK]
    sensitivity_basis = subspaces["primary_basis"][:, :SENSITIVITY_RANK]
    specifications = []

    def add(target_entry, condition, family, mode, rank, dose, delta):
        specifications.append(
            {
                "target_entry": target_entry,
                "condition": condition,
                "control_family": family,
                "mode": mode,
                "rank": int(rank),
                "dose": float(dose),
                "delta_white": np.asarray(delta, dtype=np.float64),
            }
        )

    for target_entry in STEERING_TARGETS[str(record_id)]:
        permutation = np.asarray(target_entry["permutation"], dtype=np.int64)
        primary = action_swap_delta(white, permutation, primary_basis, dose=1.0)
        sensitivity = action_swap_delta(
            white, permutation, sensitivity_basis, dose=1.0
        )
        full_swap = action_swap_delta(white, permutation, basis=None, dose=1.0)
        if min(np.linalg.norm(primary), np.linalg.norm(full_swap)) <= 1e-12:
            raise RuntimeError("targeted steering edit is degenerate")
        for dose in ACTIVE_STEERING_DOSES:
            add(
                target_entry,
                f"learned_r{PRIMARY_STEERING_RANK:03d}",
                "primary",
                "targeted_replacement",
                PRIMARY_STEERING_RANK,
                dose,
                float(dose) * primary,
            )
        add(
            target_entry,
            f"learned_r{SENSITIVITY_RANK:03d}",
            "rank_sensitivity",
            "targeted_replacement",
            SENSITIVITY_RANK,
            1.0,
            sensitivity,
        )
        shuffled = action_swap_delta(
            white,
            permutation,
            subspaces["shuffled_basis"][:, :PRIMARY_STEERING_RANK],
            dose=1.0,
        )
        add(
            target_entry,
            f"shuffled_r{PRIMARY_STEERING_RANK:03d}",
            "matched_shuffled_control",
            "targeted_replacement",
            PRIMARY_STEERING_RANK,
            1.0,
            norm_match(shuffled, primary),
        )
        for draw in range(ACTIVE_CAUSAL_RANDOM_DRAWS):
            random_delta = action_swap_delta(
                white,
                permutation,
                subspaces[f"random_basis_{draw:02d}"][:, :PRIMARY_STEERING_RANK],
                dose=1.0,
            )
            add(
                target_entry,
                f"random_r{PRIMARY_STEERING_RANK:03d}_{draw:02d}",
                "empirical_span_random_control",
                "targeted_replacement",
                PRIMARY_STEERING_RANK,
                1.0,
                norm_match(random_delta, primary),
            )
        wrong_id = int(wrong_state_map[str(record_id)])
        wrong_carrier = carrier_for_block(load_baseline(wrong_id), FIXED_BLOCK)
        wrong = wrong_state_delta(
            white,
            whiten_carrier(wrong_carrier, subspaces),
            permutation,
            primary_basis,
        )
        add(
            target_entry,
            f"wrong_state_r{PRIMARY_STEERING_RANK:03d}",
            "state_specificity_control",
            "targeted_replacement",
            PRIMARY_STEERING_RANK,
            1.0,
            norm_match(wrong, primary),
        )
        add(
            target_entry,
            f"common_mode_r{PRIMARY_STEERING_RANK:03d}",
            "matched_common_mode_control",
            "targeted_replacement",
            PRIMARY_STEERING_RANK,
            1.0,
            matched_common_mode(primary, primary_basis[:, 0]),
        )
        add(
            target_entry,
            "full_activation_swap",
            "positive_control_only",
            "targeted_replacement",
            -1,
            1.0,
            full_swap,
        )

    primary_ablation = projection_ablation_delta(white, primary_basis, dose=1.0)
    add(
        None,
        f"ablate_primary_r{PRIMARY_STEERING_RANK:03d}",
        "primary",
        "necessity",
        PRIMARY_STEERING_RANK,
        1.0,
        primary_ablation,
    )
    shuffled_ablation = projection_ablation_delta(
        white,
        subspaces["shuffled_basis"][:, :PRIMARY_STEERING_RANK],
        dose=1.0,
    )
    add(
        None,
        f"ablate_shuffled_r{PRIMARY_STEERING_RANK:03d}",
        "matched_shuffled_control",
        "necessity",
        PRIMARY_STEERING_RANK,
        1.0,
        norm_match(shuffled_ablation, primary_ablation),
    )
    for draw in range(ACTIVE_CAUSAL_RANDOM_DRAWS):
        random_ablation = projection_ablation_delta(
            white,
            subspaces[f"random_basis_{draw:02d}"][:, :PRIMARY_STEERING_RANK],
            dose=1.0,
        )
        add(
            None,
            f"ablate_random_r{PRIMARY_STEERING_RANK:03d}_{draw:02d}",
            "empirical_span_random_control",
            "necessity",
            PRIMARY_STEERING_RANK,
            1.0,
            norm_match(random_ablation, primary_ablation),
        )

    if len(specifications) != ACTIVE_INTERVENTION_FORWARDS_PER_RECORD:
        raise RuntimeError(
            f"expected {ACTIVE_INTERVENTION_FORWARDS_PER_RECORD} patched forwards, "
            f"found {len(specifications)}"
        )
    return white, specifications


def run_record_interventions(record, subspaces):
    destination = intervention_path(record["record_id"])
    if destination.exists():
        PROVENANCE_COUNTS["cache_hits"] += 1
        raise RuntimeError(f"fresh-run intervention shard already exists: {destination}")
    payload = load_baseline(record["record_id"])
    carrier = carrier_for_block(payload, FIXED_BLOCK)
    baseline_output = payload["output_eval_sketch"].astype(np.float64)
    baseline_pose = payload["decoded_pose"].astype(np.float64)
    true_cost, goal = truth_costs(record)
    white, specifications = intervention_specs(record, carrier, subspaces)
    target_entries = STEERING_TARGETS[str(int(record["record_id"]))]
    rows = [
        make_result_row(
            record,
            target_entry,
            "no_edit",
            "baseline",
            "baseline",
            0,
            0.0,
            baseline_output,
            baseline_output,
            baseline_pose,
            baseline_pose,
            true_cost,
            goal,
            0.0,
            0.0,
            0.0,
        )
        for target_entry in target_entries
    ]
    initial, actions = state_model_inputs(record["record_id"])
    for specification in specifications:
        delta_native = native_edit(specification["delta_white"], subspaces)
        delta_tensor = torch.as_tensor(delta_native, device="cuda", dtype=torch.float32)
        with torch.inference_mode():
            patched, _, _ = forward_with_carriers(
                initial,
                actions,
                PRIMARY_HORIZON,
                capture_blocks=[FIXED_BLOCK],
                intervention={"block": FIXED_BLOCK, "delta": delta_tensor},
            )
            patched_output = EVAL_OUTPUT_PROJECTOR(patched).cpu().numpy()
            patched_pose = DECODE_PHYSICAL_POSE(patched).cpu().numpy()
        entries = (
            [specification["target_entry"]]
            if specification["target_entry"] is not None
            else target_entries
        )
        for target_entry in entries:
            permutation = np.asarray(target_entry["permutation"], dtype=np.int64)
            primary_swap = action_swap_delta(
                white,
                permutation,
                subspaces["primary_basis"][:, :PRIMARY_STEERING_RANK],
                dose=1.0,
            )
            full_swap = action_swap_delta(white, permutation, basis=None, dose=1.0)
            rows.append(
                make_result_row(
                    record,
                    target_entry,
                    specification["condition"],
                    specification["control_family"],
                    specification["mode"],
                    specification["rank"],
                    specification["dose"],
                    baseline_output,
                    patched_output,
                    baseline_pose,
                    patched_pose,
                    true_cost,
                    goal,
                    np.linalg.norm(specification["delta_white"]),
                    np.linalg.norm(primary_swap),
                    np.linalg.norm(full_swap),
                )
            )
        del patched, patched_output, patched_pose, delta_tensor
    if len(rows) != ACTIVE_RESULT_ROWS_PER_RECORD:
        raise RuntimeError(
            f"expected {ACTIVE_RESULT_ROWS_PER_RECORD} result rows, found {len(rows)}"
        )
    write_json(destination, finite_json_rows(rows))
    PROVENANCE_COUNTS["intervention_generated"] += 1
    PROVENANCE_COUNTS["patched_forwards_generated"] += len(specifications)
    del initial, actions
    gc.collect()
    torch.cuda.empty_cache()
    return rows


def run_all_interventions(records):
    started = time.perf_counter()
    subspaces = load_frozen_subspaces()
    rows = []
    for index, record in enumerate(records):
        rows.extend(run_record_interventions(record, subspaces))
        write_json(
            OUT / "intervention_progress.json",
            {
                "completed": index + 1,
                "total": len(records),
                "last_record_id": int(record["record_id"]),
                "patched_forwards_generated": PROVENANCE_COUNTS["patched_forwards_generated"],
            },
        )
    TIMINGS["causal_steering_seconds"] = time.perf_counter() - started
    write_csv(EVIDENCE_DIR / "steering_state_rows.csv", rows)
    return rows


if not PIPELINE_FAILED and EVALUATION_OPENED:
    try:
        STEERING_ROWS = run_all_interventions(ALL_EVALUATION_RECORDS)
        memory_report("causal_planner_steering_complete")
    except Exception:
        record_failure("causal_planner_steering")

In [ ]:
# Apply frozen family-level representation and planner-steering gates.


def result_lookup(rows, family, record_id, target_slot, condition, key, dose=1.0):
    values = [
        row[key]
        for row in rows
        if row["action_family"] == family
        and row["record_id"] == record_id
        and row["target_slot"] == target_slot
        and row["condition"] == condition
        and np.isclose(row["dose"], dose)
    ]
    return float(values[0]) if len(values) == 1 else np.nan


def family_attempts(family):
    return [
        (int(record["record_id"]), int(record["trajectory_id"]), int(entry["target_slot"]))
        for record in FAMILY_RECORDS[family]
        for entry in STEERING_TARGETS[str(int(record["record_id"]))]
    ]


def random_median(rows, family, record_id, target_slot, key, ablate=False):
    prefix = "ablate_random" if ablate else "random"
    values = [
        result_lookup(
            rows,
            family,
            record_id,
            target_slot,
            f"{prefix}_r{PRIMARY_STEERING_RANK:03d}_{draw:02d}",
            key,
        )
        for draw in range(ACTIVE_CAUSAL_RANDOM_DRAWS)
    ]
    return float(np.nanmedian(values))


def bootstrap_interval(values, clusters, family, label):
    seed = stable_seed(BOOTSTRAP_SEED, family, label) % (2**31 - 1)
    draws = clustered_bootstrap_mean(
        np.asarray(values, dtype=np.float64),
        np.asarray(clusters, dtype=np.int64),
        ACTIVE_BOOTSTRAP_DRAWS,
        seed,
    )
    return [float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))]


def trajectory_means(values, trajectories):
    values = np.asarray(values, dtype=np.float64)
    trajectories = np.asarray(trajectories, dtype=np.int64)
    return np.asarray([
        np.mean(values[trajectories == trajectory])
        for trajectory in np.unique(trajectories)
    ])


def evaluate_family(rows, family):
    attempts = family_attempts(family)
    record_ids = [value[0] for value in attempts]
    trajectories = [value[1] for value in attempts]
    slots = [value[2] for value in attempts]

    def values(condition, key, dose=1.0):
        return np.asarray([
            result_lookup(rows, family, record_id, slot, condition, key, dose)
            for record_id, slot in zip(record_ids, slots)
        ])

    learned_name = f"learned_r{PRIMARY_STEERING_RANK:03d}"
    learned_output = values(learned_name, "output_coefficient")
    half_output = (
        values(learned_name, "output_coefficient", 0.5)
        if 0.5 in ACTIVE_STEERING_DOSES else np.full(len(attempts), np.nan)
    )
    random_output = np.asarray([
        random_median(rows, family, record_id, slot, "output_coefficient")
        for record_id, slot in zip(record_ids, slots)
    ])
    shuffled_output = values(
        f"shuffled_r{PRIMARY_STEERING_RANK:03d}", "output_coefficient"
    )
    full_output = values("full_activation_swap", "output_coefficient")
    output_gain_random = learned_output - random_output
    output_gain_shuffled = learned_output - shuffled_output

    learned_rank_gain = values(learned_name, "target_rank_gain")
    random_rank_gain = np.asarray([
        random_median(rows, family, record_id, slot, "target_rank_gain")
        for record_id, slot in zip(record_ids, slots)
    ])
    shuffled_rank_gain = values(
        f"shuffled_r{PRIMARY_STEERING_RANK:03d}", "target_rank_gain"
    )
    rank_gain_random = learned_rank_gain - random_rank_gain
    rank_gain_shuffled = learned_rank_gain - shuffled_rank_gain

    learned_choice = values(learned_name, "choice_matches_counterfactual")
    random_choice = np.asarray([
        random_median(
            rows, family, record_id, slot, "choice_matches_counterfactual"
        )
        for record_id, slot in zip(record_ids, slots)
    ])
    shuffled_choice = values(
        f"shuffled_r{PRIMARY_STEERING_RANK:03d}",
        "choice_matches_counterfactual",
    )
    full_choice = values("full_activation_swap", "choice_matches_counterfactual")
    choice_gain_random = learned_choice - random_choice
    choice_gain_shuffled = learned_choice - shuffled_choice

    learned_score = values(learned_name, "score_transfer_coefficient")
    learned_flip = values(learned_name, "choice_flipped")
    learned_physical_change = values(learned_name, "selected_true_cost_change")
    sensitivity_output = values(
        f"learned_r{SENSITIVITY_RANK:03d}", "output_coefficient"
    )

    # Necessity energy is target-independent. Use one row per record.
    unique_records = [(int(record["record_id"]), int(record["trajectory_id"])) for record in FAMILY_RECORDS[family]]
    ablate_name = f"ablate_primary_r{PRIMARY_STEERING_RANK:03d}"
    ablate_shuffled_name = f"ablate_shuffled_r{PRIMARY_STEERING_RANK:03d}"
    necessity = np.asarray([
        result_lookup(
            rows, family, record_id, 0, ablate_name,
            "output_contrast_energy_reduction",
        )
        for record_id, _ in unique_records
    ])
    necessity_shuffled = np.asarray([
        result_lookup(
            rows, family, record_id, 0, ablate_shuffled_name,
            "output_contrast_energy_reduction",
        )
        for record_id, _ in unique_records
    ])
    necessity_random = np.asarray([
        random_median(
            rows,
            family,
            record_id,
            0,
            "output_contrast_energy_reduction",
            ablate=True,
        )
        for record_id, _ in unique_records
    ])
    necessity_gain_random = necessity - necessity_random
    necessity_gain_shuffled = necessity - necessity_shuffled
    necessity_clusters = [value[1] for value in unique_records]

    finite_arrays = [
        learned_output, random_output, shuffled_output, full_output,
        learned_rank_gain, random_rank_gain, shuffled_rank_gain,
        learned_choice, random_choice, shuffled_choice, full_choice,
        learned_score, learned_flip, learned_physical_change,
        sensitivity_output, necessity, necessity_random, necessity_shuffled,
    ]
    finite = bool(all(np.all(np.isfinite(value)) for value in finite_arrays))
    output_ci = bootstrap_interval(
        output_gain_random, trajectories, family, "output_gain_random"
    )
    rank_ci = bootstrap_interval(
        rank_gain_random, trajectories, family, "rank_gain_random"
    )
    choice_ci = bootstrap_interval(
        choice_gain_random, trajectories, family, "choice_gain_random"
    )
    necessity_ci = bootstrap_interval(
        necessity_gain_random,
        necessity_clusters,
        family,
        "necessity_gain_random",
    )
    trajectory_output_gain = trajectory_means(output_gain_random, trajectories)
    trajectory_rank_gain = trajectory_means(rank_gain_random, trajectories)

    representation_pass = bool(
        finite
        and np.mean(full_output) >= MIN_FULL_SWAP_COEFFICIENT
        and np.mean(learned_output) >= MIN_PRIMARY_OUTPUT_COEFFICIENT
        and np.mean(output_gain_random) >= MIN_OUTPUT_GAIN_OVER_RANDOM
        and np.mean(output_gain_shuffled) > 0
        and np.mean(necessity) >= MIN_NECESSITY_REDUCTION
        and np.mean(necessity_gain_random) >= MIN_NECESSITY_GAIN_OVER_RANDOM
        and np.mean(necessity_gain_shuffled) >= MIN_NECESSITY_GAIN_OVER_SHUFFLED
        and (output_ci[0] > 0 if RUN_MODE == "pilot" else True)
        and (necessity_ci[0] > 0 if RUN_MODE == "pilot" else True)
        and (
            RUN_MODE == "smoke"
            or (
                np.mean(learned_output - half_output) > 0
                and exact_positive_sign_test(trajectory_output_gain)["p_value"] <= 0.05
            )
        )
    )
    planner_pass = bool(
        finite
        and np.mean(full_choice) >= MIN_FULL_TARGET_CHOICE_RATE
        and np.mean(rank_gain_random) >= MIN_TARGET_RANK_GAIN_OVER_RANDOM
        and np.mean(rank_gain_shuffled) > 0
        and np.mean(choice_gain_random) >= MIN_CHOICE_MATCH_GAIN_OVER_RANDOM
        and np.mean(choice_gain_shuffled) > 0
        and (rank_ci[0] > 0 if RUN_MODE == "pilot" else True)
        and (choice_ci[0] > 0 if RUN_MODE == "pilot" else True)
        and (
            exact_positive_sign_test(trajectory_rank_gain)["p_value"] <= 0.05
            if RUN_MODE == "pilot" else True
        )
    )
    return {
        "action_family": family,
        "trajectories": len(unique_records),
        "target_attempts": len(attempts),
        "all_required_metrics_finite": finite,
        "mean_full_output_coefficient": float(np.mean(full_output)),
        "mean_learned_output_coefficient_r128": float(np.mean(learned_output)),
        "mean_learned_output_coefficient_r64": float(np.mean(sensitivity_output)),
        "mean_random_output_coefficient": float(np.mean(random_output)),
        "mean_shuffled_output_coefficient": float(np.mean(shuffled_output)),
        "mean_output_gain_over_random": float(np.mean(output_gain_random)),
        "output_gain_over_random_ci95": output_ci,
        "output_gain_over_random_sign_test_by_trajectory": exact_positive_sign_test(trajectory_output_gain),
        "mean_full_counterfactual_choice_rate": float(np.mean(full_choice)),
        "mean_learned_counterfactual_choice_rate": float(np.mean(learned_choice)),
        "mean_random_counterfactual_choice_rate": float(np.mean(random_choice)),
        "mean_shuffled_counterfactual_choice_rate": float(np.mean(shuffled_choice)),
        "mean_choice_match_gain_over_random": float(np.mean(choice_gain_random)),
        "choice_match_gain_over_random_ci95": choice_ci,
        "mean_learned_target_rank_gain": float(np.mean(learned_rank_gain)),
        "mean_random_target_rank_gain": float(np.mean(random_rank_gain)),
        "mean_shuffled_target_rank_gain": float(np.mean(shuffled_rank_gain)),
        "mean_target_rank_gain_over_random": float(np.mean(rank_gain_random)),
        "target_rank_gain_over_random_ci95": rank_ci,
        "target_rank_gain_over_random_sign_test_by_trajectory": exact_positive_sign_test(trajectory_rank_gain),
        "mean_planner_score_transfer_coefficient": float(np.mean(learned_score)),
        "mean_learned_choice_flip_rate": float(np.mean(learned_flip)),
        "mean_selected_true_cost_change": float(np.mean(learned_physical_change)),
        "mean_necessity_energy_reduction": float(np.mean(necessity)),
        "mean_necessity_random_reduction": float(np.mean(necessity_random)),
        "mean_necessity_shuffled_reduction": float(np.mean(necessity_shuffled)),
        "mean_necessity_gain_over_random": float(np.mean(necessity_gain_random)),
        "necessity_gain_over_random_ci95": necessity_ci,
        "representation_gate_pass": representation_pass,
        "planner_steering_gate_pass": planner_pass,
        "causal_planner_chain_gate_pass": bool(representation_pass and planner_pass),
    }


def fresh_run_certificate():
    expected = {
        "truth_generated": len(POOL_SPECS),
        "baseline_generated": len(ALL_EVALUATION_RECORDS),
        "intervention_generated": len(ALL_EVALUATION_RECORDS),
        "patched_forwards_generated": len(ALL_EVALUATION_RECORDS)
        * ACTIVE_INTERVENTION_FORWARDS_PER_RECORD,
        "cache_hits": 0,
    }
    passed = bool(not OUT_PREEXISTED and PROVENANCE_COUNTS == expected)
    payload = {
        "out_preexisted": bool(OUT_PREEXISTED),
        "fresh_run_required": bool(FRESH_RUN_REQUIRED),
        "observed_counts": dict(PROVENANCE_COUNTS),
        "expected_counts": expected,
        "passed": passed,
    }
    write_json(OUT / "fresh_run_certificate.json", payload)
    return payload


if PIPELINE_FAILED:
    DECISION_PAYLOAD = {"status": "INCONCLUSIVE", "failure": FAILURE_MESSAGE}
elif not EVALUATION_OPENED:
    DECISION_PAYLOAD = {"status": "INCONCLUSIVE", "reason": "steering targets were not frozen"}
else:
    try:
        FAMILY_RESULTS = {
            family: evaluate_family(STEERING_ROWS, family)
            for family in TRANSFER_FAMILIES
        }
        FRESH_CERTIFICATE = fresh_run_certificate()
        representation_families = [
            family for family in TRANSFER_FAMILIES
            if FAMILY_RESULTS[family]["representation_gate_pass"]
        ]
        planner_families = [
            family for family in TRANSFER_FAMILIES
            if FAMILY_RESULTS[family]["causal_planner_chain_gate_pass"]
        ]
        if RUN_MODE == "smoke":
            candidate_status = "SMOKE_ONLY"
        elif len(planner_families) == len(TRANSFER_FAMILIES):
            candidate_status = "CONFIRMED_CAUSAL_PLANNER_STEERING_BOTH_FAMILIES"
        elif planner_families:
            candidate_status = "PARTIAL_CAUSAL_PLANNER_STEERING"
        elif len(representation_families) == len(TRANSFER_FAMILIES):
            candidate_status = "PREDICTION_MEDIATOR_TRANSFER_WITHOUT_CONFIRMED_PLANNER_STEERING"
        else:
            candidate_status = "NO_CONFIRMED_STAGE20_CAUSAL_CHAIN"
        source_eligible = bool(SOURCE_IDENTITY.get("confirmation_eligible", False))
        prior_eligible = bool(PRIOR_ARTIFACTS_VALIDATED)
        confirmation_eligible = bool(
            source_eligible and prior_eligible and FRESH_CERTIFICATE["passed"]
        )
        status = (
            candidate_status
            if RUN_MODE == "smoke" or confirmation_eligible
            else "UNBOUND_NONFRESH_OR_WRONG_PRIOR_EXPLORATORY_RESULT"
        )
        DECISION_PAYLOAD = {
            "status": status,
            "candidate_status": candidate_status,
            "source_bound_claim_eligible": source_eligible,
            "prior_artifacts_claim_eligible": prior_eligible,
            "fresh_run_claim_eligible": FRESH_CERTIFICATE["passed"],
            "confirmation_eligible": confirmation_eligible,
            "representation_passed_families": representation_families,
            "planner_chain_passed_families": planner_families,
            "family_results": FAMILY_RESULTS,
            "claim_boundary": {
                "nonvisual_evaluation_only": True,
                "steering_targets_are_near_frontier_baseline_actions": True,
                "targets_selected_without_simulator_outcomes": True,
                "physical_improvement_claim_authorized": False,
                "multi_step_closed_loop_control_tested": False,
                "other_models_or_environments_generalized": False,
                "causal_claim_is_prediction_to_ranking_to_choice_mediation": True,
            },
        }
        write_json(OUT / "stage20_decision.json", DECISION_PAYLOAD)
        print(json.dumps(DECISION_PAYLOAD, indent=2))
    except Exception:
        record_failure("decision")
        DECISION_PAYLOAD = {"status": "INCONCLUSIVE", "failure": FAILURE_MESSAGE}

if not (OUT / "stage20_decision.json").exists():
    write_json(OUT / "stage20_decision.json", DECISION_PAYLOAD)

In [ ]:
# Package compact audit evidence, including hashes for excluded raw shards.
write_json(OUT / "timings.json", TIMINGS)
memory_report("final")
if not PIPELINE_FAILED:
    (OUT / "FAILURE_TRACE.txt").write_text("NONE\n")

raw_roots = [TRUTH_DIR, BASELINE_DIR, INTERVENTION_DIR]
raw_files = [
    path for root in raw_roots for path in sorted(root.rglob("*")) if path.is_file()
]
raw_files += [
    path for path in sorted(SUBSPACE_DIR.rglob("*.npz")) if path.is_file()
]
raw_manifest = [
    {"path": str(path.relative_to(OUT)), "bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in raw_files
]
write_json(OUT / "raw_shard_manifest.json", raw_manifest)

excluded_roots = {ASSET_DIR, TRUTH_DIR, BASELINE_DIR, INTERVENTION_DIR}
compact_files = []
for path in sorted(OUT.rglob("*")):
    if not path.is_file():
        continue
    if any(root == path or root in path.parents for root in excluded_roots):
        continue
    if SUBSPACE_DIR in path.parents and path.suffix == ".npz":
        continue
    if path.name.startswith("stage20_causal_planner_steering_result_bundle_"):
        continue
    compact_files.append(path)

manifest = [
    {"path": str(path.relative_to(OUT)), "bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in compact_files
]
write_json(OUT / "result_zip_manifest.json", manifest)
compact_files.append(OUT / "result_zip_manifest.json")

staging = OUT / "_result_staging"
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()
for path in compact_files:
    relative = path.relative_to(OUT)
    destination = staging / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, destination)

archive_base = OUT / f"stage20_causal_planner_steering_result_bundle_{RUN_SIGNATURE[:12]}"
archive = Path(shutil.make_archive(str(archive_base), "zip", staging))
shutil.rmtree(staging)
print(f"RUN_STATUS: {DECISION_PAYLOAD['status']}")
print(f"RESULT_BUNDLE: {archive}")
print(f"RESULT_BUNDLE_SHA256: {sha256_file(archive)}")
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files

        files.download(str(archive))
    except Exception as error:
        print(f"Automatic download unavailable: {error}")